
# PhysioNet EEGMMIDB — FBGAN + CRNN-DF V4

V4 is the corrected research-stage adaptation of Zhang et al. (2023)
for PhysioNet EEG Motor Movement/Imagery.

Core V4 changes
----------------
1. All 109 PhysioNet subjects are available as the source population.
2. num_target_subjects_to_test=5 means five HELD-OUT TARGET FOLDS,
   not five total subjects in the training set.
3. Each target fold uses the other 104 subjects as source training data.
4. The target FBGAN is trained once per target, then its generated
   samples are reused for a controlled 0/500/1000/2000/3000 ablation.
5. The generator keeps the paper's:
      z = 1600
      FC = 256000
      ConvTranspose kernels:
          (3,15), (3,15), (3,5), (4,5), (1,2)
      strides:
          (1,3), (1,3), (1,2), (2,1), (1,1)
      first four BatchNorm layers
      LeakyReLU activations
   while a final geometry adapter maps the temporal dimension to
   the PhysioNet 640-sample window.
6. D_phi keeps the paper's discriminator sequence but adapts
   the spatial kernel from (22,1) to (64,1).
7. D_psi uses the paper's filter-bank/sparse-spatial-filter pathway
   and adapts its spatial reduction to the number of LASSO-selected
   CSP components for the target subject.
8. CRNN-DF uses:
      kernel = (64,45)
      max-pool = (1,75), stride 10
      2-layer LSTM, hidden size 64
      dropout = 0.5
9. Center loss:
      lambda = 0.1
      center shift alpha = 0.02
      center update every 15 epochs.
10. Source normalization statistics are calculated only from source
    subjects and then applied to target data.
11. The final notebook reports a controlled augmentation ablation.

Scientific interpretation
-------------------------
The original paper reports 63.52 +/- 10.70% for CRNN-DF and
72.82 +/- 10.44% after 3,000 fake samples on BCI Competition IV-2a.
Those values are NOT expected PhysioNet numbers; V4 tests transfer
of the methodology to a different dataset.

The current V3 failure was approximately chance after adding 3,000
synthetic trials to only ~350 real source trials. V4 fixes that
experimental imbalance by using the remaining 104 PhysioNet source
subjects for every target fold.

Subject selection
-----------------
A quality score is used only to define a DEVELOPMENT target cohort.
It never uses model accuracy. For a final publication evaluation,
replace this development cohort with a pre-registered or fixed
subject list and evaluate all 109 subjects.


In [1]:

# ============================================================
# CELL 1 - ENVIRONMENT, IMPORTS, AND V4 CONFIGURATION
# ============================================================

import gc
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt, welch
from sklearn.linear_model import LassoCV
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

SEED = 42


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


@dataclass
class Config:
    # --------------------------------------------------------
    # PhysioNet geometry
    # --------------------------------------------------------
    sfreq: int = 160
    epoch_seconds: float = 4.0
    n_channels: int = 64
    n_times: int = 640
    n_classes: int = 4

    # --------------------------------------------------------
    # Local dataset
    # --------------------------------------------------------
    data_path: str = (
        "/Users/ashokvarmabevara/MtechProj/eegmmidb"
    )

    # --------------------------------------------------------
    # LOSO development experiment
    #
    # 5 = five held-out target folds.
    # The source pool for each target is ALL OTHER subjects.
    # --------------------------------------------------------
    num_target_subjects_to_test: int = 5
    quality_scan_limit: int = 109

    # --------------------------------------------------------
    # Target selection
    # --------------------------------------------------------
    use_quality_ranked_development_targets: bool = True

    # --------------------------------------------------------
    # Target adaptation protocol
    #
    # True  = paper-faithful target adaptation.
    # False = target calibration/test split.
    # --------------------------------------------------------
    paper_faithful_target_adaptation: bool = True
    target_calibration_fraction: float = 0.20

    # --------------------------------------------------------
    # Source validation
    # --------------------------------------------------------
    source_validation_fraction: float = 0.10

    # --------------------------------------------------------
    # CRNN-DF
    # --------------------------------------------------------
    classifier_batch_size: int = 32
    classifier_lr: float = 1e-4
    classifier_epochs: int = 80
    classifier_patience: int = 12
    classifier_weight_decay: float = 1e-4
    classifier_gradient_clip: float = 1.0

    # --------------------------------------------------------
    # FBGAN
    # --------------------------------------------------------
    gan_batch_size: int = 8
    gan_lr: float = 1e-4
    gan_epochs: int = 25
    latent_dim: int = 1600

    # We generate the maximum amount once per target.
    # Lower augmentation levels reuse subsets from this pool.
    max_fake_total: int = 3000

    # Real/fake GAN labels.
    real_label: float = 0.90
    fake_label: float = 0.00

    # D_phi/D_psi and G update schedule.
    discriminator_steps: int = 1
    generator_steps: int = 1

    # Optional sparse-feature matching stabilizer.
    sparse_match_weight: float = 0.25

    # --------------------------------------------------------
    # FBCSP + LASSO
    # --------------------------------------------------------
    lasso_cv: int = 5
    max_sparse_features: int = 160

    # --------------------------------------------------------
    # Center/discriminative feature learning
    # --------------------------------------------------------
    lambda_center: float = 0.1
    center_update_every: int = 15
    center_alpha: float = 0.02

    # --------------------------------------------------------
    # Controlled augmentation ablation.
    # These numbers are TOTAL fake samples, balanced across
    # the four classes.
    # --------------------------------------------------------
    augmentation_levels: tuple = (
        0,
        500,
        1000,
        2000,
        3000,
    )

    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------
    num_workers: int = 0
    debug_shapes: bool = False


CFG = Config()

BANDS = [
    (1, 4),
    (4, 8),
    (8, 12),
    (12, 16),
    (16, 20),
    (20, 24),
    (24, 28),
    (28, 32),
    (32, 35),
    (35, 38),
]

CLASS_NAMES = {
    0: "Left Fist",
    1: "Right Fist",
    2: "Both Fists",
    3: "Both Feet",
}

print("=" * 70)
print("V4 CONFIGURATION")
print("=" * 70)
print("Device:", DEVICE)
print("Dataset:", CFG.data_path)
print("Targets:", CFG.num_target_subjects_to_test)
print("Source subjects per fold: all other available subjects")
print("Augmentation:", CFG.augmentation_levels)
print("Latent dimension:", CFG.latent_dim)
print("Max fake total:", CFG.max_fake_total)


V4 CONFIGURATION
Device: mps
Dataset: /Users/ashokvarmabevara/MtechProj/eegmmidb
Targets: 5
Source subjects per fold: all other available subjects
Augmentation: (0, 500, 1000, 2000, 3000)
Latent dimension: 1600
Max fake total: 3000



# ============================================================
# CELL 2 - V4 ARCHITECTURE INFOGRAPHICS
# ============================================================

# ============================================================
# HYBRID LOSO
# ============================================================
#
#                 PhysioNet 109 subjects
#                         |
#            +------------+------------+
#            |                         |
#       104 SOURCE                1 TARGET
#       SUBJECTS                   SUBJECT
#            |                         |
#            |                    target EEG
#            |                         |
#            |                 10-band FBCSP
#            |                         |
#            |                    OVR-CSP
#            |                         |
#            |                       LASSO
#            |                         |
#            |                   sparse filters
#            |                         |
#            |                  +------+------+
#            |                  |             |
#            |                D_phi         D_psi
#            |                  |             |
#            |                  +------+------+
#            |                         |
#            |                target-specific
#            |                  FBGAN samples
#            |                         |
#            +-----------+-------------+
#                        |
#               0/500/1000/2000/3000
#                generated samples
#                        |
#                        v
#                  CRNN-DF training
#                        |
#                 Center Distance Loss
#                        |
#                  held-out target
#
#
# ============================================================
# GENERATOR
# ============================================================
#
# z = (B,1600)
#       |
#       v
# FC = 256000
#       |
# reshape = (B,128,H0,W0)
#       |
# ConvTrans1: 128 -> 128
# kernel (3,15), stride (1,3)
# BatchNorm + LeakyReLU
#       |
# ConvTrans2: 128 -> 128
# kernel (3,15), stride (1,3)
# BatchNorm + LeakyReLU
#       |
# ConvTrans3: 128 -> 64
# kernel (3,5), stride (1,2)
# BatchNorm + LeakyReLU
#       |
# ConvTrans4: 64 -> 32
# kernel (4,5), stride (2,1)
# BatchNorm + LeakyReLU
#       |
# ConvTrans5: 32 -> 1
# kernel (1,2), stride (1,1)
#       |
# geometry adapter
#       |
#       v
# (B,1,64,640)
#
#
# ============================================================
# D_phi
# ============================================================
#
# EEG (B,1,64,640)
#       |
# Conv1 (1,23)
#       |
# Conv2 spatial (64,1)
#       |
# Conv3 (1,17)
#       |
# MaxPool (1,6), stride 6
#       |
# Conv4 (1,7), stride 7
#       |
# MaxPool (1,6), stride 6
#       |
# Flatten
#       |
# FC -> 1
#
#
# ============================================================
# D_psi
# ============================================================
#
# Sparse-FB EEG
# (B,1,K,640), K = selected LASSO components
#       |
# Conv1 (1,23)
#       |
# Conv2 (4,1), stride (4,1)
#       |
# Conv3 (K',1)  # collapses selected spatial dimension
#       |
# Conv4 (1,17)
#       |
# MaxPool (1,6)
#       |
# Conv5 (1,7)
#       |
# MaxPool (1,6)
#       |
# FC -> 1
#
#
# ============================================================
# CRNN-DF
# ============================================================
#
# EEG (B,1,64,640)
#       |
# Conv2D kernel (64,45)
#       |
# MaxPool (1,75), stride 10
#       |
# Sequence (B,T,40)
#       |
# 2-layer LSTM
# hidden=64
#       |
# discriminative feature v_i (B,64)
#       |
# CrossEntropy + 0.1 * center distance
#       |
# 4-class logits
#
print("V4 architecture diagrams loaded.")


In [2]:
# ============================================================
# CELL 3 - LOCAL PHYSIONET EEGMMIDB DATA LOADER
# ============================================================
# This notebook uses the already-downloaded PhysioNet EEGMMIDB
# dataset directly from the local Mac filesystem.
#
# Expected structure:
#
# /Users/ashokvarmabevara/MtechProj/eegmmidb/
# ├── S001/
# │   ├── S001R04.edf
# │   ├── S001R06.edf
# │   ├── S001R08.edf
# │   ├── S001R10.edf
# │   ├── S001R12.edf
# │   └── S001R14.edf
# ├── S002/
# └── ...
# ============================================================

from pathlib import Path

import mne
import numpy as np
from scipy.signal import resample


# ============================================================
# DATASET CONFIGURATION
# ============================================================

DATASET_ROOT = Path(
    "/Users/ashokvarmabevara/MtechProj/eegmmidb"
)

TARGET_FS = 160
TRIAL_DURATION = 4.0

N_CHANNELS = 64
N_SAMPLES = int(TARGET_FS * TRIAL_DURATION)

LEFT_RIGHT_RUNS = [4, 8, 12]
HANDS_FEET_RUNS = [6, 10, 14]

ALL_RUNS = (
    LEFT_RIGHT_RUNS
    + HANDS_FEET_RUNS
)

CLASS_NAMES = {
    0: "Left Fist",
    1: "Right Fist",
    2: "Both Fists",
    3: "Both Feet",
}


# ============================================================
# DATASET VALIDATION
# ============================================================

def check_dataset_structure():
    """Validate the local EEGMMIDB dataset structure."""

    print("=" * 70)
    print("LOCAL PHYSIONET EEGMMIDB DATASET CHECK")
    print("=" * 70)

    print(f"Dataset path: {DATASET_ROOT}")
    print(f"Dataset exists: {DATASET_ROOT.exists()}")

    if not DATASET_ROOT.exists():
        raise FileNotFoundError(
            "Dataset folder was not found:\n"
            f"{DATASET_ROOT}"
        )

    subject_dirs = sorted(
        [
            path
            for path in DATASET_ROOT.iterdir()
            if path.is_dir()
            and path.name.startswith("S")
        ]
    )

    print(
        f"\nNumber of subject folders found: "
        f"{len(subject_dirs)}"
    )

    if len(subject_dirs) == 0:
        raise RuntimeError(
            "No subject folders were found in the dataset."
        )

    print("\nFirst 10 subject folders:")

    for subject_dir in subject_dirs[:10]:
        print(subject_dir.name)

    print("=" * 70)

    return subject_dirs


# ============================================================
# EDF FILE RESOLUTION
# ============================================================

def get_edf_file(subject_id, run_id):
    """Return the EDF path for one subject and one run."""

    subject_folder = (
        DATASET_ROOT
        / f"S{subject_id:03d}"
    )

    expected_file = (
        subject_folder
        / f"S{subject_id:03d}R{run_id:02d}.edf"
    )

    if expected_file.exists():
        return expected_file

    candidates = list(
        subject_folder.glob(
            f"*R{run_id:02d}.edf"
        )
    )

    if candidates:
        return candidates[0]

    return None


# ============================================================
# LOAD ONE EDF RUN
# ============================================================

def load_single_run(subject_id, run_id):
    """
    Load one local PhysioNet EEGMMIDB EDF recording.

    The data is resampled to 160 Hz if necessary.
    """

    edf_path = get_edf_file(
        subject_id=subject_id,
        run_id=run_id,
    )

    if edf_path is None:
        print(
            f"[WARNING] Missing EDF file: "
            f"S{subject_id:03d} R{run_id:02d}"
        )
        return None

    try:
        raw = mne.io.read_raw_edf(
            edf_path,
            preload=True,
            verbose=False,
        )

        original_fs = float(
            raw.info["sfreq"]
        )

        if not np.isclose(
            original_fs,
            TARGET_FS,
        ):
            print(
                f"Resampling S{subject_id:03d} "
                f"R{run_id:02d}: "
                f"{original_fs:.2f} Hz -> "
                f"{TARGET_FS} Hz"
            )

            raw.resample(
                TARGET_FS,
                npad="auto",
                verbose=False,
            )

        return raw

    except Exception as error:
        print(
            f"[ERROR] Failed to load "
            f"S{subject_id:03d} "
            f"R{run_id:02d}"
        )
        print(error)

        return None


# ============================================================
# RUN-SPECIFIC LABEL MAPPING
# ============================================================

def get_class_mapping(run_id):
    """
    Map PhysioNet T1/T2 annotations to the unified 4 classes.

    Runs 4, 8, 12:
        T1 -> Left Fist
        T2 -> Right Fist

    Runs 6, 10, 14:
        T1 -> Both Fists
        T2 -> Both Feet
    """

    if run_id in LEFT_RIGHT_RUNS:
        return {
            "T1": 0,
            "T2": 1,
        }

    if run_id in HANDS_FEET_RUNS:
        return {
            "T1": 2,
            "T2": 3,
        }

    raise ValueError(
        f"Unsupported run: {run_id}"
    )


# ============================================================
# EXTRACT FIXED 4-SECOND TRIALS
# ============================================================

def extract_trials_from_run(
    raw,
    subject_id,
    run_id,
):
    """
    Extract motor-imagery trials.

    Output:
        X -> (N_trials, 64, 640)
        y -> (N_trials,)
    """

    class_mapping = get_class_mapping(
        run_id
    )

    try:
        events, event_id = (
            mne.events_from_annotations(
                raw,
                verbose=False,
            )
        )

    except Exception as error:
        print(
            f"[WARNING] Annotation extraction failed "
            f"for S{subject_id:03d} "
            f"R{run_id:02d}"
        )
        print(error)

        return None, None

    annotation_by_code = {
        code: name
        for name, code in event_id.items()
    }

    selected_events = []

    for event in events:
        annotation = annotation_by_code.get(
            event[2]
        )

        if annotation in class_mapping:
            selected_events.append(event)

    if not selected_events:
        print(
            f"[WARNING] No valid motor-imagery "
            f"events found for "
            f"S{subject_id:03d} "
            f"R{run_id:02d}"
        )

        return None, None

    selected_events = np.asarray(
        selected_events,
        dtype=int,
    )

    selected_event_id = {
        name: code
        for name, code in event_id.items()
        if name in class_mapping
    }

    try:
        epochs = mne.Epochs(
            raw,
            selected_events,
            event_id=selected_event_id,
            tmin=0.0,
            tmax=(
                TRIAL_DURATION
                - 1.0 / TARGET_FS
            ),
            baseline=None,
            preload=True,
            picks="eeg",
            verbose=False,
        )

        X = epochs.get_data(
            copy=True
        )

    except Exception as error:
        print(
            f"[ERROR] Epoch extraction failed "
            f"for S{subject_id:03d} "
            f"R{run_id:02d}"
        )
        print(error)

        return None, None

    labels = []

    for event in epochs.events:
        annotation = annotation_by_code.get(
            event[2]
        )

        if annotation in class_mapping:
            labels.append(
                class_mapping[annotation]
            )

    y = np.asarray(
        labels,
        dtype=np.int64,
    )

    # Defensive shape handling.
    if X.shape[1] != N_CHANNELS:
        print(
            f"[WARNING] Expected {N_CHANNELS} EEG channels "
            f"but found {X.shape[1]} for "
            f"S{subject_id:03d} "
            f"R{run_id:02d}. Skipping run."
        )

        return None, None

    if X.shape[2] != N_SAMPLES:
        X = resample(
            X,
            N_SAMPLES,
            axis=2,
        )

    X = X.astype(
        np.float32,
        copy=False,
    )

    print(
        f"S{subject_id:03d} "
        f"R{run_id:02d} | "
        f"X={X.shape} | "
        f"Class counts="
        f"{np.bincount(y, minlength=4)}"
    )

    return X, y


# ============================================================
# LOAD ALL SIX MI RUNS FOR ONE SUBJECT
# ============================================================

def load_subject_data(subject_id):
    """
    Load runs 4, 6, 8, 10, 12, and 14 for one subject.

    Returns:
        X_subject: (N_trials, 64, 640)
        y_subject: (N_trials,)
    """

    all_trials = []
    all_labels = []

    print("\n" + "=" * 70)
    print(
        f"LOADING SUBJECT S{subject_id:03d}"
    )
    print("=" * 70)

    for run_id in ALL_RUNS:
        raw = load_single_run(
            subject_id=subject_id,
            run_id=run_id,
        )

        if raw is None:
            continue

        X_run, y_run = (
            extract_trials_from_run(
                raw=raw,
                subject_id=subject_id,
                run_id=run_id,
            )
        )

        if X_run is None:
            continue

        all_trials.append(X_run)
        all_labels.append(y_run)

    if not all_trials:
        print(
            f"[WARNING] No valid trials found "
            f"for S{subject_id:03d}"
        )

        return None, None

    X_subject = np.concatenate(
        all_trials,
        axis=0,
    )

    y_subject = np.concatenate(
        all_labels,
        axis=0,
    )

    print("\nSubject summary")
    print(
        f"Subject: S{subject_id:03d}"
    )
    print(
        f"EEG shape: {X_subject.shape}"
    )
    print(
        f"Class distribution: "
        f"{np.bincount(y_subject, minlength=4)}"
    )

    return X_subject, y_subject


# ============================================================
# LOAD MULTIPLE SUBJECTS
# ============================================================

def load_dataset(
    subject_ids,
):
    """
    Load a collection of subjects.

    Returns:
        X_all
        y_all
        subject_ids_per_trial
    """

    X_parts = []
    y_parts = []
    subject_parts = []

    for subject_id in subject_ids:
        try:
            X_subject, y_subject = (
                load_subject_data(
                    subject_id
                )
            )

            if X_subject is None:
                continue

            X_parts.append(X_subject)
            y_parts.append(y_subject)

            subject_parts.append(
                np.full(
                    len(y_subject),
                    subject_id,
                    dtype=np.int64,
                )
            )

        except Exception as error:
            print(
                f"[ERROR] Skipping "
                f"S{subject_id:03d}:"
            )
            print(error)

    if not X_parts:
        raise RuntimeError(
            "No valid subjects were loaded."
        )

    X_all = np.concatenate(
        X_parts,
        axis=0,
    )

    y_all = np.concatenate(
        y_parts,
        axis=0,
    )

    subject_ids_per_trial = (
        np.concatenate(
            subject_parts,
            axis=0,
        )
    )

    print("\n" + "=" * 70)
    print("DATASET LOADING COMPLETE")
    print("=" * 70)
    print(
        f"X shape: {X_all.shape}"
    )
    print(
        f"y shape: {y_all.shape}"
    )
    print(
        f"Loaded subjects: "
        f"{np.unique(subject_ids_per_trial)}"
    )

    return (
        X_all,
        y_all,
        subject_ids_per_trial,
    )


# ============================================================

# INITIAL DATASET VALIDATION
# ============================================================

subject_directories = (
    check_dataset_structure()
)

print("\nLocal dataset loader is ready.")
print(
    "Dataset loading for the LOSO experiment "
    "is performed in the next cell."
)


LOCAL PHYSIONET EEGMMIDB DATASET CHECK
Dataset path: /Users/ashokvarmabevara/MtechProj/eegmmidb
Dataset exists: True

Number of subject folders found: 109

First 10 subject folders:
S001
S002
S003
S004
S005
S006
S007
S008
S009
S010

Local dataset loader is ready.
Dataset loading for the LOSO experiment is performed in the next cell.



## V4 loader note

The local PhysioNet dataset is loaded directly from:

`/Users/ashokvarmabevara/MtechProj/eegmmidb`

The LOSO experiment does **not** load only five subjects. Five is the
number of held-out target folds. For each target, every other available
subject is loaded into the source training pool.


In [3]:
# ============================================================
# CELL 3 - LOCAL PHYSIONET EEGMMIDB DATA LOADER
# ============================================================
# This notebook uses the already-downloaded PhysioNet EEGMMIDB
# dataset directly from the local Mac filesystem.
#
# Expected structure:
#
# /Users/ashokvarmabevara/MtechProj/eegmmidb/
# ├── S001/
# │   ├── S001R04.edf
# │   ├── S001R06.edf
# │   ├── S001R08.edf
# │   ├── S001R10.edf
# │   ├── S001R12.edf
# │   └── S001R14.edf
# ├── S002/
# └── ...
# ============================================================

from pathlib import Path

import mne
import numpy as np
from scipy.signal import resample


# ============================================================
# DATASET CONFIGURATION
# ============================================================

DATASET_ROOT = Path(
    "/Users/ashokvarmabevara/MtechProj/eegmmidb"
)

TARGET_FS = 160
TRIAL_DURATION = 4.0

N_CHANNELS = 64
N_SAMPLES = int(TARGET_FS * TRIAL_DURATION)

LEFT_RIGHT_RUNS = [4, 8, 12]
HANDS_FEET_RUNS = [6, 10, 14]

ALL_RUNS = (
    LEFT_RIGHT_RUNS
    + HANDS_FEET_RUNS
)

CLASS_NAMES = {
    0: "Left Fist",
    1: "Right Fist",
    2: "Both Fists",
    3: "Both Feet",
}


# ============================================================
# DATASET VALIDATION
# ============================================================

def check_dataset_structure():
    """Validate the local EEGMMIDB dataset structure."""

    print("=" * 70)
    print("LOCAL PHYSIONET EEGMMIDB DATASET CHECK")
    print("=" * 70)

    print(f"Dataset path: {DATASET_ROOT}")
    print(f"Dataset exists: {DATASET_ROOT.exists()}")

    if not DATASET_ROOT.exists():
        raise FileNotFoundError(
            "Dataset folder was not found:\n"
            f"{DATASET_ROOT}"
        )

    subject_dirs = sorted(
        [
            path
            for path in DATASET_ROOT.iterdir()
            if path.is_dir()
            and path.name.startswith("S")
        ]
    )

    print(
        f"\nNumber of subject folders found: "
        f"{len(subject_dirs)}"
    )

    if len(subject_dirs) == 0:
        raise RuntimeError(
            "No subject folders were found in the dataset."
        )

    print("\nFirst 10 subject folders:")

    for subject_dir in subject_dirs[:10]:
        print(subject_dir.name)

    print("=" * 70)

    return subject_dirs


# ============================================================
# EDF FILE RESOLUTION
# ============================================================

def get_edf_file(subject_id, run_id):
    """Return the EDF path for one subject and one run."""

    subject_folder = (
        DATASET_ROOT
        / f"S{subject_id:03d}"
    )

    expected_file = (
        subject_folder
        / f"S{subject_id:03d}R{run_id:02d}.edf"
    )

    if expected_file.exists():
        return expected_file

    candidates = list(
        subject_folder.glob(
            f"*R{run_id:02d}.edf"
        )
    )

    if candidates:
        return candidates[0]

    return None


# ============================================================
# LOAD ONE EDF RUN
# ============================================================

def load_single_run(subject_id, run_id):
    """
    Load one local PhysioNet EEGMMIDB EDF recording.

    The data is resampled to 160 Hz if necessary.
    """

    edf_path = get_edf_file(
        subject_id=subject_id,
        run_id=run_id,
    )

    if edf_path is None:
        print(
            f"[WARNING] Missing EDF file: "
            f"S{subject_id:03d} R{run_id:02d}"
        )
        return None

    try:
        raw = mne.io.read_raw_edf(
            edf_path,
            preload=True,
            verbose=False,
        )

        original_fs = float(
            raw.info["sfreq"]
        )

        if not np.isclose(
            original_fs,
            TARGET_FS,
        ):
            print(
                f"Resampling S{subject_id:03d} "
                f"R{run_id:02d}: "
                f"{original_fs:.2f} Hz -> "
                f"{TARGET_FS} Hz"
            )

            raw.resample(
                TARGET_FS,
                npad="auto",
                verbose=False,
            )

        return raw

    except Exception as error:
        print(
            f"[ERROR] Failed to load "
            f"S{subject_id:03d} "
            f"R{run_id:02d}"
        )
        print(error)

        return None


# ============================================================
# RUN-SPECIFIC LABEL MAPPING
# ============================================================

def get_class_mapping(run_id):
    """
    Map PhysioNet T1/T2 annotations to the unified 4 classes.

    Runs 4, 8, 12:
        T1 -> Left Fist
        T2 -> Right Fist

    Runs 6, 10, 14:
        T1 -> Both Fists
        T2 -> Both Feet
    """

    if run_id in LEFT_RIGHT_RUNS:
        return {
            "T1": 0,
            "T2": 1,
        }

    if run_id in HANDS_FEET_RUNS:
        return {
            "T1": 2,
            "T2": 3,
        }

    raise ValueError(
        f"Unsupported run: {run_id}"
    )


# ============================================================
# EXTRACT FIXED 4-SECOND TRIALS
# ============================================================

def extract_trials_from_run(
    raw,
    subject_id,
    run_id,
):
    """
    Extract motor-imagery trials.

    Output:
        X -> (N_trials, 64, 640)
        y -> (N_trials,)
    """

    class_mapping = get_class_mapping(
        run_id
    )

    try:
        events, event_id = (
            mne.events_from_annotations(
                raw,
                verbose=False,
            )
        )

    except Exception as error:
        print(
            f"[WARNING] Annotation extraction failed "
            f"for S{subject_id:03d} "
            f"R{run_id:02d}"
        )
        print(error)

        return None, None

    annotation_by_code = {
        code: name
        for name, code in event_id.items()
    }

    selected_events = []

    for event in events:
        annotation = annotation_by_code.get(
            event[2]
        )

        if annotation in class_mapping:
            selected_events.append(event)

    if not selected_events:
        print(
            f"[WARNING] No valid motor-imagery "
            f"events found for "
            f"S{subject_id:03d} "
            f"R{run_id:02d}"
        )

        return None, None

    selected_events = np.asarray(
        selected_events,
        dtype=int,
    )

    selected_event_id = {
        name: code
        for name, code in event_id.items()
        if name in class_mapping
    }

    try:
        epochs = mne.Epochs(
            raw,
            selected_events,
            event_id=selected_event_id,
            tmin=0.0,
            tmax=(
                TRIAL_DURATION
                - 1.0 / TARGET_FS
            ),
            baseline=None,
            preload=True,
            picks="eeg",
            verbose=False,
        )

        X = epochs.get_data(
            copy=True
        )

    except Exception as error:
        print(
            f"[ERROR] Epoch extraction failed "
            f"for S{subject_id:03d} "
            f"R{run_id:02d}"
        )
        print(error)

        return None, None

    labels = []

    for event in epochs.events:
        annotation = annotation_by_code.get(
            event[2]
        )

        if annotation in class_mapping:
            labels.append(
                class_mapping[annotation]
            )

    y = np.asarray(
        labels,
        dtype=np.int64,
    )

    # Defensive shape handling.
    if X.shape[1] != N_CHANNELS:
        print(
            f"[WARNING] Expected {N_CHANNELS} EEG channels "
            f"but found {X.shape[1]} for "
            f"S{subject_id:03d} "
            f"R{run_id:02d}. Skipping run."
        )

        return None, None

    if X.shape[2] != N_SAMPLES:
        X = resample(
            X,
            N_SAMPLES,
            axis=2,
        )

    X = X.astype(
        np.float32,
        copy=False,
    )

    print(
        f"S{subject_id:03d} "
        f"R{run_id:02d} | "
        f"X={X.shape} | "
        f"Class counts="
        f"{np.bincount(y, minlength=4)}"
    )

    return X, y


# ============================================================
# LOAD ALL SIX MI RUNS FOR ONE SUBJECT
# ============================================================

def load_subject_data(subject_id):
    """
    Load runs 4, 6, 8, 10, 12, and 14 for one subject.

    Returns:
        X_subject: (N_trials, 64, 640)
        y_subject: (N_trials,)
    """

    all_trials = []
    all_labels = []

    print("\n" + "=" * 70)
    print(
        f"LOADING SUBJECT S{subject_id:03d}"
    )
    print("=" * 70)

    for run_id in ALL_RUNS:
        raw = load_single_run(
            subject_id=subject_id,
            run_id=run_id,
        )

        if raw is None:
            continue

        X_run, y_run = (
            extract_trials_from_run(
                raw=raw,
                subject_id=subject_id,
                run_id=run_id,
            )
        )

        if X_run is None:
            continue

        all_trials.append(X_run)
        all_labels.append(y_run)

    if not all_trials:
        print(
            f"[WARNING] No valid trials found "
            f"for S{subject_id:03d}"
        )

        return None, None

    X_subject = np.concatenate(
        all_trials,
        axis=0,
    )

    y_subject = np.concatenate(
        all_labels,
        axis=0,
    )

    print("\nSubject summary")
    print(
        f"Subject: S{subject_id:03d}"
    )
    print(
        f"EEG shape: {X_subject.shape}"
    )
    print(
        f"Class distribution: "
        f"{np.bincount(y_subject, minlength=4)}"
    )

    return X_subject, y_subject


# ============================================================
# LOAD MULTIPLE SUBJECTS
# ============================================================

def load_dataset(
    subject_ids,
):
    """
    Load a collection of subjects.

    Returns:
        X_all
        y_all
        subject_ids_per_trial
    """

    X_parts = []
    y_parts = []
    subject_parts = []

    for subject_id in subject_ids:
        try:
            X_subject, y_subject = (
                load_subject_data(
                    subject_id
                )
            )

            if X_subject is None:
                continue

            X_parts.append(X_subject)
            y_parts.append(y_subject)

            subject_parts.append(
                np.full(
                    len(y_subject),
                    subject_id,
                    dtype=np.int64,
                )
            )

        except Exception as error:
            print(
                f"[ERROR] Skipping "
                f"S{subject_id:03d}:"
            )
            print(error)

    if not X_parts:
        raise RuntimeError(
            "No valid subjects were loaded."
        )

    X_all = np.concatenate(
        X_parts,
        axis=0,
    )

    y_all = np.concatenate(
        y_parts,
        axis=0,
    )

    subject_ids_per_trial = (
        np.concatenate(
            subject_parts,
            axis=0,
        )
    )

    print("\n" + "=" * 70)
    print("DATASET LOADING COMPLETE")
    print("=" * 70)
    print(
        f"X shape: {X_all.shape}"
    )
    print(
        f"y shape: {y_all.shape}"
    )
    print(
        f"Loaded subjects: "
        f"{np.unique(subject_ids_per_trial)}"
    )

    return (
        X_all,
        y_all,
        subject_ids_per_trial,
    )


# ============================================================

# INITIAL DATASET VALIDATION
# ============================================================

subject_directories = (
    check_dataset_structure()
)

print("\nLocal dataset loader is ready.")
print(
    "Dataset loading for the LOSO experiment "
    "is performed in the next cell."
)


LOCAL PHYSIONET EEGMMIDB DATASET CHECK
Dataset path: /Users/ashokvarmabevara/MtechProj/eegmmidb
Dataset exists: True

Number of subject folders found: 109

First 10 subject folders:
S001
S002
S003
S004
S005
S006
S007
S008
S009
S010

Local dataset loader is ready.
Dataset loading for the LOSO experiment is performed in the next cell.


In [4]:

# ============================================================
# CELL 3B - OBJECTIVE DEVELOPMENT TARGET SELECTION
# ============================================================

DATASET_ROOT = Path(CFG.data_path)

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset does not exist: {DATASET_ROOT}"
    )


def summarize_subject_quality(subject_id):
    """
    Objective data-quality summary.

    Model accuracy is never used in this ranking.
    """

    x_subject, y_subject = load_subject_data(
        subject_id
    )

    if x_subject is None:
        return None

    counts = np.bincount(
        y_subject,
        minlength=CFG.n_classes,
    )

    mean_count = max(
        float(counts.mean()),
        1.0,
    )

    balance_score = (
        1.0
        - float(counts.std())
        / mean_count
    )

    balance_score = float(
        np.clip(
            balance_score,
            0.0,
            1.0,
        )
    )

    trial_score = float(
        np.clip(
            len(y_subject)
            / 450.0,
            0.0,
            1.0,
        )
    )

    trial_std = np.std(
        x_subject,
        axis=(1, 2),
    )

    nonflat_score = float(
        1.0
        - np.mean(
            trial_std < 1e-8
        )
    )

    abs_values = np.abs(
        x_subject
    )

    median_abs = np.median(
        abs_values
    )

    extreme_ratio = float(
        np.mean(
            abs_values
            > 20.0 * max(
                median_abs,
                1e-8,
            )
        )
    )

    artifact_score = float(
        np.clip(
            1.0
            - 5.0
            * extreme_ratio,
            0.0,
            1.0,
        )
    )

    quality_score = (
        0.35 * balance_score
        + 0.30 * trial_score
        + 0.20 * nonflat_score
        + 0.15 * artifact_score
    )

    return {
        "subject": int(subject_id),
        "n_trials": int(len(y_subject)),
        "left_fist": int(counts[0]),
        "right_fist": int(counts[1]),
        "both_fists": int(counts[2]),
        "both_feet": int(counts[3]),
        "balance_score": balance_score,
        "trial_score": trial_score,
        "nonflat_score": nonflat_score,
        "artifact_score": artifact_score,
        "quality_score": quality_score,
    }


available_subject_ids = sorted(
    int(path.name[1:])
    for path in DATASET_ROOT.iterdir()
    if path.is_dir()
    and path.name.startswith("S")
    and path.name[1:].isdigit()
)

if len(available_subject_ids) < (
    CFG.num_target_subjects_to_test + 1
):
    raise RuntimeError(
        "Not enough valid subjects for V4 LOSO."
    )

quality_rows = []

print(
    f"Scanning {len(available_subject_ids)} "
    "subjects for development-quality ranking..."
)

for subject_id in tqdm(
    available_subject_ids[
        :CFG.quality_scan_limit
    ]
):
    try:
        result = summarize_subject_quality(
            subject_id
        )

        if result is not None:
            quality_rows.append(
                result
            )

    except Exception as error:
        print(
            f"[QUALITY WARNING] "
            f"S{subject_id:03d}: {error}"
        )

    gc.collect()

QUALITY_TABLE = pd.DataFrame(
    quality_rows
)

if QUALITY_TABLE.empty:
    raise RuntimeError(
        "Quality ranking found no valid subjects."
    )

QUALITY_TABLE = (
    QUALITY_TABLE
    .sort_values(
        "quality_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("\nTOP QUALITY SUBJECTS")
print(
    QUALITY_TABLE.head(15).to_string(
        index=False
    )
)

if CFG.use_quality_ranked_development_targets:
    TARGET_SUBJECTS = (
        QUALITY_TABLE
        .head(
            CFG.num_target_subjects_to_test
        )["subject"]
        .astype(int)
        .tolist()
    )
else:
    TARGET_SUBJECTS = (
        available_subject_ids[
            :CFG.num_target_subjects_to_test
        ]
    )

# All remaining subjects are source subjects for each fold.
ALL_SUBJECTS = sorted(
    set(available_subject_ids)
)

print(
    "\nDEVELOPMENT TARGET SUBJECTS:",
    TARGET_SUBJECTS,
)

print(
    "TOTAL AVAILABLE SUBJECTS:",
    len(ALL_SUBJECTS),
)

print(
    "SOURCE SUBJECTS PER TARGET:",
    len(ALL_SUBJECTS) - 1,
)

if len(ALL_SUBJECTS) >= 109:
    print(
        "PhysioNet 109-subject population detected."
    )

print(
    "\nImportant:"
    "\n  num_target_subjects_to_test=5 means "
    "five target folds."
    "\n  It does NOT restrict source training to five subjects."
)


Scanning 109 subjects for development-quality ranking...


  0%|          | 0/109 [00:00<?, ?it/s]


LOADING SUBJECT S001
S001 R04 | X=(15, 64, 640) | Class counts=[8 7 0 0]
S001 R08 | X=(15, 64, 640) | Class counts=[8 7 0 0]
S001 R12 | X=(15, 64, 640) | Class counts=[7 8 0 0]
S001 R06 | X=(15, 64, 640) | Class counts=[0 0 7 8]
S001 R10 | X=(15, 64, 640) | Class counts=[0 0 7 8]
S001 R14 | X=(15, 64, 640) | Class counts=[0 0 7 8]

Subject summary
Subject: S001
EEG shape: (90, 64, 640)
Class distribution: [23 22 21 24]

LOADING SUBJECT S002
S002 R04 | X=(15, 64, 640) | Class counts=[7 8 0 0]
S002 R08 | X=(15, 64, 640) | Class counts=[8 7 0 0]
S002 R12 | X=(15, 64, 640) | Class counts=[8 7 0 0]
S002 R06 | X=(15, 64, 640) | Class counts=[0 0 8 7]
S002 R10 | X=(15, 64, 640) | Class counts=[0 0 8 7]
S002 R14 | X=(15, 64, 640) | Class counts=[0 0 8 7]

Subject summary
Subject: S002
EEG shape: (90, 64, 640)
Class distribution: [23 22 24 21]

LOADING SUBJECT S003
S003 R04 | X=(15, 64, 640) | Class counts=[8 7 0 0]
S003 R08 | X=(15, 64, 640) | Class counts=[7 8 0 0]
S003 R12 | X=(15, 64, 640)

In [5]:
# CELL 4 - Preprocessing + OVR-FBCSP + LASSO
# ============================================================


def butter_bandpass(
    data,
    low,
    high,
    sfreq,
    order=5,
):
    """Apply the paper's fifth-order Butterworth bandpass."""

    nyquist = sfreq / 2.0

    low_norm = max(
        low / nyquist,
        1e-5,
    )
    high_norm = min(
        high / nyquist,
        0.999,
    )

    sos = butter(
        order,
        [low_norm, high_norm],
        btype="bandpass",
        output="sos",
    )

    return sosfiltfilt(
        sos,
        data,
        axis=-1,
    ).astype(np.float32)


class TrainZScore:
    """
    Training-only channel-wise z-score.

    The mean/std are estimated only on source training subjects.
    """

    def fit(self, x):
        self.mean_ = x.mean(
            axis=(0, 2),
            keepdims=True,
        )
        self.std_ = np.sqrt(
            x.var(
                axis=(0, 2),
                keepdims=True,
            )
            + 1e-8
        )
        return self

    def transform(self, x):
        return (
            (x - self.mean_)
            / self.std_
        ).astype(np.float32)

    def fit_transform(self, x):
        return self.fit(x).transform(x)


def normalized_covariance(trial):
    covariance = trial @ trial.T
    trace = np.trace(covariance)
    return (
        covariance
        / (trace + 1e-10)
    )


class OVRFBCSP:
    """
    One-vs-rest CSP for 10 sub-bands.

    Four classes x four retained eigenvectors =
    16 candidate components per band.

    10 bands x 16 = 160 candidate features.
    """

    def __init__(
        self,
        bands=BANDS,
        n_vectors_per_class=4,
        reg=1e-6,
    ):
        self.bands = bands
        self.n_vectors_per_class = (
            n_vectors_per_class
        )
        self.reg = reg
        self.filters_ = []

    def fit(self, x, y, sfreq=160):
        self.filters_ = []

        for low, high in self.bands:
            x_band = butter_bandpass(
                x,
                low,
                high,
                sfreq,
            )

            band_filters = []

            for class_id in range(4):
                positive = x_band[
                    y == class_id
                ]
                negative = x_band[
                    y != class_id
                ]

                if (
                    len(positive) == 0
                    or len(negative) == 0
                ):
                    continue

                r_pos = np.mean(
                    [
                        normalized_covariance(
                            trial
                        )
                        for trial in positive
                    ],
                    axis=0,
                )

                r_neg = np.mean(
                    [
                        normalized_covariance(
                            trial
                        )
                        for trial in negative
                    ],
                    axis=0,
                )

                composite = (
                    r_pos
                    + r_neg
                    + self.reg
                    * np.eye(
                        x.shape[1]
                    )
                )

                eigenvalues, eigenvectors = (
                    np.linalg.eigh(
                        np.linalg.pinv(composite)
                        @ r_pos
                    )
                )

                # The paper retains the four largest eigenvalue
                # eigenvectors for every one-vs-rest spatial filter.
                order = np.argsort(
                    eigenvalues
                )[::-1]

                selected = order[
                    : self.n_vectors_per_class
                ]

                band_filters.append(
                    eigenvectors[
                        :, selected
                    ].T
                )

            if band_filters:
                self.filters_.append(
                    np.concatenate(
                        band_filters,
                        axis=0,
                    ).astype(np.float32)
                )
            else:
                self.filters_.append(
                    np.zeros(
                        (
                            16,
                            x.shape[1],
                        ),
                        dtype=np.float32,
                    )
                )

        return self

    def transform(self, x, sfreq=160):
        features = []

        for (
            low,
            high,
        ), spatial_filters in zip(
            self.bands,
            self.filters_,
        ):
            x_band = butter_bandpass(
                x,
                low,
                high,
                sfreq,
            )

            projected = np.einsum(
                "fc,nct->nft",
                spatial_filters,
                x_band,
            )

            variance = np.var(
                projected,
                axis=-1,
            )

            features.append(
                np.log(
                    variance + 1e-10
                )
            )

        return np.concatenate(
            features,
            axis=1,
        ).astype(np.float32)


class SparseFBCSP:
    """
    LASSO-selected FBCSP.

    To prevent the previous 151-159/160 near-dense representation,
    we keep the strongest LASSO components up to max_sparse_features.
    The selected spatial filters are then used by D_psi.
    """

    def __init__(
        self,
        bands=BANDS,
        max_sparse_features=64,
        random_state=SEED,
    ):
        self.bands = bands
        self.max_sparse_features = (
            max_sparse_features
        )
        self.random_state = random_state

        self.csp = OVRFBCSP(
            bands=bands
        )

        self.scaler = StandardScaler()

        self.selected_indices_ = None
        self.coefficient_strength_ = None

    def fit(self, x, y, sfreq=160):
        self.csp.fit(
            x,
            y,
            sfreq,
        )

        features = self.csp.transform(
            x,
            sfreq,
        )

        scaled = self.scaler.fit_transform(
            features
        )

        y_one_hot = np.eye(4)[y]
        coefficient_strength = np.zeros(
            features.shape[1],
            dtype=np.float64,
        )

        class_counts = np.bincount(
            y,
            minlength=4,
        )

        cv_splits = max(
            2,
            min(
                self.random_state and 5 or 5,
                int(class_counts.min()),
            ),
        )

        for class_id in range(4):
            model = LassoCV(
                cv=cv_splits,
                random_state=self.random_state,
                max_iter=20000,
                n_alphas=100,
                eps=1e-4,
            )

            model.fit(
                scaled,
                y_one_hot[:, class_id],
            )

            coefficient_strength += (
                np.abs(model.coef_)
            )

        self.coefficient_strength_ = (
            coefficient_strength
        )

        nonzero = np.flatnonzero(
            coefficient_strength > 1e-10
        )

        if len(nonzero) == 0:
            selected = np.argsort(
                coefficient_strength
            )[
                -min(
                    self.max_sparse_features,
                    len(coefficient_strength),
                ):
            ]
        elif len(nonzero) > self.max_sparse_features:
            selected = nonzero[
                np.argsort(
                    coefficient_strength[
                        nonzero
                    ]
                )[
                    -self.max_sparse_features:
                ]
            ]
        else:
            selected = nonzero

        self.selected_indices_ = np.sort(
            selected
        )

        return self

    def transform(self, x, sfreq=160):
        features = self.csp.transform(
            x,
            sfreq,
        )

        scaled = self.scaler.transform(
            features
        )

        return scaled[
            :,
            self.selected_indices_,
        ].astype(np.float32)

    def selected_filter_metadata(self):
        """
        Return selected CSP filters grouped by source band.

        Each selected item is:
            (global_index, band_index, filter_vector)
        """

        filters = []

        concatenated = np.concatenate(
            self.csp.filters_,
            axis=0,
        )

        for index in self.selected_indices_:
            band_index = (
                int(index) // 16
            )

            filters.append(
                (
                    int(index),
                    band_index,
                    concatenated[index].astype(
                        np.float32
                    ),
                )
            )

        return filters


def filter_bank_tensor(
    x,
    bands=BANDS,
    sfreq=160,
):
    """Return (N, 10, 64, 640)."""

    bank = [
        butter_bandpass(
            x,
            low,
            high,
            sfreq,
        )
        for low, high in bands
    ]

    return np.stack(
        bank,
        axis=1,
    ).astype(np.float32)


def fit_sparse_fbcsp(
    x,
    y,
    cfg=CFG,
):
    model = SparseFBCSP(
        max_sparse_features=(
            cfg.max_sparse_features
        )
    )

    model.fit(
        x,
        y,
        cfg.sfreq,
    )

    return model


In [6]:

# ============================================================
# CELL 5 - V4 FBGAN: PAPER-PARAMETER GENERATOR + D_phi + D_psi
# ============================================================

# This cell is self-contained.
# Run it once after Cell 4 and before the training cell.
#
# The generator keeps the paper's published:
#   FC 1600 -> 256000
#   ConvTrans:
#       (3,15)/(1,3)
#       (3,15)/(1,3)
#       (3,5)/(1,2)
#       (4,5)/(2,1)
#       (1,2)/(1,1)
#
# The FC output is reshaped to 128 x 25 x 80 = 256000.
# Because the exact original paper geometry is for 22 x 1000,
# the PhysioNet temporal geometry is explicitly adapted to 64 x 640
# with a final interpolation adapter.
#
# The final generator layer is linear rather than tanh so the
# synthetic EEG is not artificially clipped to [-1,1] after
# source-only z-score normalization.
# ============================================================


class FBGANGenerator(nn.Module):
    """
    Paper-parameter generator adapted to PhysioNet.

    Input:
        z = (B,1600)

    Internal:
        FC -> 256000 = 128*25*80

    Output:
        (B,1,64,640)
    """

    def __init__(
        self,
        latent_dim=1600,
    ):
        super().__init__()

        self.fc = nn.Linear(
            latent_dim,
            256000,
        )

        self.deconv1 = nn.ConvTranspose2d(
            128,
            128,
            kernel_size=(3, 15),
            stride=(1, 3),
        )

        self.bn1 = nn.BatchNorm2d(
            128
        )

        self.deconv2 = nn.ConvTranspose2d(
            128,
            128,
            kernel_size=(3, 15),
            stride=(1, 3),
        )

        self.bn2 = nn.BatchNorm2d(
            128
        )

        self.deconv3 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=(3, 5),
            stride=(1, 2),
        )

        self.bn3 = nn.BatchNorm2d(
            64
        )

        self.deconv4 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=(4, 5),
            stride=(2, 1),
        )

        self.bn4 = nn.BatchNorm2d(
            32
        )

        self.deconv5 = nn.ConvTranspose2d(
            32,
            1,
            kernel_size=(1, 2),
            stride=(1, 1),
        )

        self.activation = nn.LeakyReLU(
            0.2,
            inplace=True,
        )

    def forward(
        self,
        z,
        debug=False,
    ):
        x = self.fc(
            z
        )

        x = x.view(
            -1,
            128,
            25,
            80,
        )

        if debug:
            print(
                "G FC reshape:",
                tuple(x.shape),
            )

        x = self.deconv1(
            x
        )
        x = self.bn1(
            x
        )
        x = self.activation(
            x
        )

        if debug:
            print(
                "G ConvTrans1:",
                tuple(x.shape),
            )

        x = self.deconv2(
            x
        )
        x = self.bn2(
            x
        )
        x = self.activation(
            x
        )

        if debug:
            print(
                "G ConvTrans2:",
                tuple(x.shape),
            )

        x = self.deconv3(
            x
        )
        x = self.bn3(
            x
        )
        x = self.activation(
            x
        )

        if debug:
            print(
                "G ConvTrans3:",
                tuple(x.shape),
            )

        x = self.deconv4(
            x
        )
        x = self.bn4(
            x
        )
        x = self.activation(
            x
        )

        if debug:
            print(
                "G ConvTrans4:",
                tuple(x.shape),
            )

        x = self.deconv5(
            x
        )

        if debug:
            print(
                "G ConvTrans5:",
                tuple(x.shape),
            )

        # Explicit PhysioNet geometry adapter.
        if (
            x.shape[-2] != 64
            or x.shape[-1] != 640
        ):
            x = F.interpolate(
                x,
                size=(64, 640),
                mode="bilinear",
                align_corners=False,
            )

        if debug:
            print(
                "G PhysioNet output:",
                tuple(x.shape),
            )

        return x


class RawEEGDiscriminator(nn.Module):
    """
    D_phi.

    Published sequence:
        Conv1 (1,23)
        Conv2 spatial (C,1)
        Conv3 (1,17)
        MaxPool (1,6), stride 6
        Conv4 (1,7), stride 7
        MaxPool (1,6), stride 6
        FC

    PhysioNet:
        C = 64

    For 640 samples, the no-padding temporal geometry produces
    a final 30 x 1 x 2 map, hence FC input = 60.
    """

    def __init__(
        self,
        n_times=640,
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            1,
            10,
            kernel_size=(1, 23),
            stride=(1, 1),
        )

        self.conv2 = nn.Conv2d(
            10,
            30,
            kernel_size=(64, 1),
            stride=(1, 1),
        )

        self.conv3 = nn.Conv2d(
            30,
            30,
            kernel_size=(1, 17),
            stride=(1, 1),
        )

        self.pool1 = nn.MaxPool2d(
            kernel_size=(1, 6),
            stride=(1, 6),
        )

        self.conv4 = nn.Conv2d(
            30,
            30,
            kernel_size=(1, 7),
            stride=(1, 7),
        )

        self.pool2 = nn.MaxPool2d(
            kernel_size=(1, 6),
            stride=(1, 6),
        )

        final_times = (
            n_times - 23 + 1
        )

        final_times = (
            final_times - 17 + 1
        )

        final_times = (
            final_times - 6
        ) // 6 + 1

        final_times = (
            final_times - 7
        ) // 7 + 1

        final_times = (
            final_times - 6
        ) // 6 + 1

        if final_times <= 0:
            raise ValueError(
                "Invalid D_phi temporal geometry."
            )

        self.fc = nn.Linear(
            30 * final_times,
            1,
        )

        self.act = nn.LeakyReLU(
            0.2,
            inplace=True,
        )

    def forward(
        self,
        x,
        debug=False,
    ):
        x = self.act(
            self.conv1(x)
        )

        if debug:
            print(
                "D_phi Conv1:",
                tuple(x.shape),
            )

        x = self.act(
            self.conv2(x)
        )

        if debug:
            print(
                "D_phi Conv2:",
                tuple(x.shape),
            )

        x = self.act(
            self.conv3(x)
        )

        if debug:
            print(
                "D_phi Conv3:",
                tuple(x.shape),
            )

        x = self.pool1(
            x
        )

        if debug:
            print(
                "D_phi Pool1:",
                tuple(x.shape),
            )

        x = self.act(
            self.conv4(x)
        )

        if debug:
            print(
                "D_phi Conv4:",
                tuple(x.shape),
            )

        x = self.pool2(
            x
        )

        if debug:
            print(
                "D_phi Pool2:",
                tuple(x.shape),
            )

        x = torch.flatten(
            x,
            start_dim=1,
        )

        if debug:
            print(
                "D_phi Flatten:",
                tuple(x.shape),
            )

        return self.fc(
            x
        )


class SparseFilterBankDiscriminator(nn.Module):
    """
    D_psi.

    The paper uses:
        Conv1 (1,23)
        Conv2 (4,1), stride (4,1)
        Conv3 (Var,1)
        Conv4 (1,17)
        MaxPool (1,6)
        Conv5 (1,7)
        MaxPool (1,6)
        FC

    Here Var is determined directly from the number K of
    selected LASSO spatial-filter components.
    """

    def __init__(
        self,
        n_sparse_channels,
        n_times=640,
    ):
        super().__init__()

        if n_sparse_channels < 4:
            raise ValueError(
                "D_psi requires at least 4 sparse channels."
            )

        self.conv1 = nn.Conv2d(
            1,
            10,
            kernel_size=(1, 23),
            stride=(1, 1),
        )

        self.conv2 = nn.Conv2d(
            10,
            30,
            kernel_size=(4, 1),
            stride=(4, 1),
        )

        reduced_channels = (
            n_sparse_channels - 4
        ) // 4 + 1

        if reduced_channels <= 0:
            raise ValueError(
                "Invalid D_psi spatial geometry."
            )

        self.conv3 = nn.Conv2d(
            30,
            30,
            kernel_size=(
                reduced_channels,
                1,
            ),
            stride=(1, 1),
        )

        self.conv4 = nn.Conv2d(
            30,
            30,
            kernel_size=(1, 17),
            stride=(1, 1),
        )

        self.pool1 = nn.MaxPool2d(
            kernel_size=(1, 6),
            stride=(1, 6),
        )

        self.conv5 = nn.Conv2d(
            30,
            30,
            kernel_size=(1, 7),
            stride=(1, 1),
        )

        self.pool2 = nn.MaxPool2d(
            kernel_size=(1, 6),
            stride=(1, 6),
        )

        # Temporal geometry.
        final_times = (
            n_times - 23 + 1
        )

        final_times = (
            final_times - 17 + 1
        )

        final_times = (
            final_times - 6
        ) // 6 + 1

        final_times = (
            final_times - 7 + 1
        )

        final_times = (
            final_times - 6
        ) // 6 + 1

        if final_times <= 0:
            raise ValueError(
                "Invalid D_psi temporal geometry."
            )

        self.fc = nn.Linear(
            30 * final_times,
            1,
        )

        self.act = nn.LeakyReLU(
            0.2,
            inplace=True,
        )

    def forward(
        self,
        x,
        debug=False,
    ):
        x = self.act(
            self.conv1(x)
        )

        if debug:
            print(
                "D_psi Conv1:",
                tuple(x.shape),
            )

        x = self.act(
            self.conv2(x)
        )

        if debug:
            print(
                "D_psi Conv2:",
                tuple(x.shape),
            )

        x = self.act(
            self.conv3(x)
        )

        if debug:
            print(
                "D_psi Conv3:",
                tuple(x.shape),
            )

        x = self.act(
            self.conv4(x)
        )

        if debug:
            print(
                "D_psi Conv4:",
                tuple(x.shape),
            )

        x = self.pool1(
            x
        )

        if debug:
            print(
                "D_psi Pool1:",
                tuple(x.shape),
            )

        x = self.act(
            self.conv5(x)
        )

        if debug:
            print(
                "D_psi Conv5:",
                tuple(x.shape),
            )

        x = self.pool2(
            x
        )

        if debug:
            print(
                "D_psi Pool2:",
                tuple(x.shape),
            )

        x = torch.flatten(
            x,
            start_dim=1,
        )

        if debug:
            print(
                "D_psi Flatten:",
                tuple(x.shape),
            )

        return self.fc(
            x
        )


# ============================================================
# DIFFERENTIABLE FILTER BANK
# ============================================================


def differentiable_filter_bank(
    x,
    bands=BANDS,
    sfreq=160,
):
    squeezed = x.squeeze(1)

    n_times = (
        squeezed.shape[-1]
    )

    frequencies = (
        torch.fft.rfftfreq(
            n_times,
            d=1.0 / sfreq,
            device=x.device,
        )
    )

    spectrum = torch.fft.rfft(
        squeezed,
        dim=-1,
    )

    outputs = []

    for low, high in bands:
        mask = (
            (frequencies >= low)
            & (frequencies <= high)
        ).to(
            spectrum.dtype
        )

        outputs.append(
            torch.fft.irfft(
                spectrum * mask,
                n=n_times,
                dim=-1,
            )
        )

    return torch.stack(
        outputs,
        dim=1,
    )


def sparse_csp_filterbank_torch(
    x,
    sparse_fbcsp,
):
    """
    Return exactly K sparse CSP channels.

    Output:
        (B,1,K,640)
    """

    filtered = (
        differentiable_filter_bank(
            x,
            BANDS,
            CFG.sfreq,
        )
    )

    metadata = (
        sparse_fbcsp
        .selected_filter_metadata()
    )

    maps = []

    for (
        _global_idx,
        band_idx,
        filter_vector,
    ) in metadata:

        weight = torch.as_tensor(
            filter_vector,
            device=x.device,
            dtype=x.dtype,
        )

        projected = torch.einsum(
            "c,bct->bt",
            weight,
            filtered[
                :,
                band_idx,
            ],
        )

        maps.append(
            projected
        )

    if not maps:
        raise RuntimeError(
            "LASSO returned zero sparse filters."
        )

    sparse_maps = torch.stack(
        maps,
        dim=1,
    )

    return sparse_maps.unsqueeze(
        1
    )


def sparse_log_variance_torch(
    sparse_map,
):
    return torch.log(
        torch.var(
            sparse_map.squeeze(1),
            dim=-1,
        )
        + 1e-8
    )


# ============================================================
# TRAIN ONE CLASS-SPECIFIC FBGAN
# ============================================================


def train_one_class_fbgan(
    real_class_x,
    sparse_fbcsp,
    cfg,
):
    n_sparse = len(
        sparse_fbcsp.selected_indices_
    )

    if n_sparse < 4:
        raise RuntimeError(
            "Too few LASSO components for D_psi: "
            f"{n_sparse}"
        )

    print(
        "Real class data:",
        real_class_x.shape,
    )

    print(
        "Sparse components:",
        n_sparse,
    )

    generator = (
        FBGANGenerator(
            cfg.latent_dim
        )
        .to(DEVICE)
    )

    d_phi = (
        RawEEGDiscriminator(
            n_times=cfg.n_times
        )
        .to(DEVICE)
    )

    d_psi = (
        SparseFilterBankDiscriminator(
            n_sparse_channels=n_sparse,
            n_times=cfg.n_times,
        )
        .to(DEVICE)
    )

    g_optimizer = torch.optim.Adam(
        generator.parameters(),
        lr=cfg.gan_lr,
        betas=(0.5, 0.999),
    )

    d_phi_optimizer = torch.optim.Adam(
        d_phi.parameters(),
        lr=cfg.gan_lr,
        betas=(0.5, 0.999),
    )

    d_psi_optimizer = torch.optim.Adam(
        d_psi.parameters(),
        lr=cfg.gan_lr,
        betas=(0.5, 0.999),
    )

    criterion = nn.BCEWithLogitsLoss()

    real_tensor = torch.tensor(
        real_class_x,
        dtype=torch.float32,
    )

    loader = DataLoader(
        TensorDataset(
            real_tensor
        ),
        batch_size=cfg.gan_batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=0,
    )

    history = []

    for epoch in range(
        1,
        cfg.gan_epochs + 1,
    ):
        generator.train()
        d_phi.train()
        d_psi.train()

        d_phi_loss_epoch = 0.0
        d_psi_loss_epoch = 0.0
        g_loss_epoch = 0.0
        sparse_loss_epoch = 0.0
        batches = 0

        for (
            real_batch_cpu,
        ) in loader:

            real_batch = (
                real_batch_cpu
                .to(DEVICE)
                .unsqueeze(1)
            )

            batch_size = (
                real_batch.shape[0]
            )

            # =================================================
            # D updates
            # =================================================

            for _ in range(
                cfg.discriminator_steps
            ):
                z = torch.randn(
                    batch_size,
                    cfg.latent_dim,
                    device=DEVICE,
                )

                with torch.no_grad():
                    fake_batch = generator(
                        z,
                        debug=False,
                    )

                # D_phi
                d_phi_optimizer.zero_grad(
                    set_to_none=True
                )

                real_phi = d_phi(
                    real_batch
                )

                fake_phi = d_phi(
                    fake_batch.detach()
                )

                real_target = torch.full_like(
                    real_phi,
                    cfg.real_label,
                )

                fake_target = torch.full_like(
                    fake_phi,
                    cfg.fake_label,
                )

                d_phi_loss = (
                    criterion(
                        real_phi,
                        real_target,
                    )
                    + criterion(
                        fake_phi,
                        fake_target,
                    )
                ) * 0.5

                d_phi_loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    d_phi.parameters(),
                    max_norm=5.0,
                )

                d_phi_optimizer.step()

                # D_psi
                d_psi_optimizer.zero_grad(
                    set_to_none=True
                )

                real_sparse = (
                    sparse_csp_filterbank_torch(
                        real_batch,
                        sparse_fbcsp,
                    )
                )

                fake_sparse = (
                    sparse_csp_filterbank_torch(
                        fake_batch.detach(),
                        sparse_fbcsp,
                    )
                )

                real_psi = d_psi(
                    real_sparse
                )

                fake_psi = d_psi(
                    fake_sparse
                )

                d_psi_loss = (
                    criterion(
                        real_psi,
                        real_target,
                    )
                    + criterion(
                        fake_psi,
                        fake_target,
                    )
                ) * 0.5

                d_psi_loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    d_psi.parameters(),
                    max_norm=5.0,
                )

                d_psi_optimizer.step()

            # =================================================
            # G update
            # =================================================

            for _ in range(
                cfg.generator_steps
            ):
                g_optimizer.zero_grad(
                    set_to_none=True
                )

                z = torch.randn(
                    batch_size,
                    cfg.latent_dim,
                    device=DEVICE,
                )

                fake_batch = generator(
                    z,
                    debug=False,
                )

                fake_phi = d_phi(
                    fake_batch
                )

                fake_sparse = (
                    sparse_csp_filterbank_torch(
                        fake_batch,
                        sparse_fbcsp,
                    )
                )

                fake_psi = d_psi(
                    fake_sparse
                )

                generator_target_phi = (
                    torch.ones_like(
                        fake_phi
                    )
                )

                generator_target_psi = (
                    torch.ones_like(
                        fake_psi
                    )
                )

                loss_g_phi = criterion(
                    fake_phi,
                    generator_target_phi,
                )

                loss_g_psi = criterion(
                    fake_psi,
                    generator_target_psi,
                )

                # Sparse feature distribution matching.
                real_sparse_stats = (
                    sparse_log_variance_torch(
                        real_sparse
                    )
                    .mean(
                        dim=0,
                        keepdim=True,
                    )
                )

                fake_sparse_stats = (
                    sparse_log_variance_torch(
                        fake_sparse
                    )
                    .mean(
                        dim=0,
                        keepdim=True,
                    )
                )

                sparse_match = F.l1_loss(
                    fake_sparse_stats,
                    real_sparse_stats.detach(),
                )

                g_loss = (
                    loss_g_phi
                    + loss_g_psi
                    + cfg.sparse_match_weight
                    * sparse_match
                )

                g_loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    generator.parameters(),
                    max_norm=5.0,
                )

                g_optimizer.step()

            d_phi_loss_epoch += (
                d_phi_loss.item()
            )

            d_psi_loss_epoch += (
                d_psi_loss.item()
            )

            g_loss_epoch += (
                g_loss.item()
            )

            sparse_loss_epoch += (
                sparse_match.item()
            )

            batches += 1

            del (
                z,
                fake_batch,
                real_phi,
                fake_phi,
                real_psi,
                fake_psi,
                real_sparse,
                fake_sparse,
            )

        denom = max(
            batches,
            1,
        )

        row = {
            "epoch": epoch,
            "d_phi": d_phi_loss_epoch / denom,
            "d_psi": d_psi_loss_epoch / denom,
            "g": g_loss_epoch / denom,
            "sparse_match": (
                sparse_loss_epoch / denom
            ),
        }

        history.append(
            row
        )

        if (
            epoch == 1
            or epoch % 5 == 0
            or epoch == cfg.gan_epochs
        ):
            print(
                f"GAN Epoch {epoch:03d}/"
                f"{cfg.gan_epochs} | "
                f"D_phi={row['d_phi']:.4f} | "
                f"D_psi={row['d_psi']:.4f} | "
                f"G={row['g']:.4f} | "
                f"Sparse={row['sparse_match']:.4f}"
            )

        if DEVICE.type == "mps":
            torch.mps.empty_cache()

    return (
        generator,
        d_phi,
        d_psi,
        pd.DataFrame(
            history
        ),
    )


@torch.no_grad()
def generate_fake_samples(
    generator,
    n_samples,
    cfg,
):
    generator.eval()

    chunks = []
    remaining = int(
        n_samples
    )

    while remaining > 0:
        current = min(
            16,
            remaining,
        )

        z = torch.randn(
            current,
            cfg.latent_dim,
            device=DEVICE,
        )

        fake = generator(
            z,
            debug=False,
        )

        expected = (
            current,
            1,
            cfg.n_channels,
            cfg.n_times,
        )

        if tuple(
            fake.shape
        ) != expected:
            raise RuntimeError(
                "Generator output mismatch: "
                f"expected {expected}, "
                f"got {tuple(fake.shape)}"
            )

        chunks.append(
            fake.squeeze(1)
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        remaining -= current

        del (
            z,
            fake,
        )

        if DEVICE.type == "mps":
            torch.mps.empty_cache()

    return np.concatenate(
        chunks,
        axis=0,
    )


def generate_balanced_fake_pool(
    target_x,
    target_y,
    sparse_fbcsp,
    cfg,
):
    """
    Train one GAN per target class and generate 3,000 total
    balanced samples.

    Later ablation levels reuse this maximum pool.
    """

    total = cfg.max_fake_total

    if total % cfg.n_classes != 0:
        raise ValueError(
            "max_fake_total must be divisible by number of classes."
        )

    per_class = (
        total
        // cfg.n_classes
    )

    fake_parts = []
    fake_labels = []
    histories = []

    for class_id in range(
        cfg.n_classes
    ):
        print("\n" + "-" * 70)
        print(
            f"FBGAN TARGET CLASS "
            f"{class_id}: "
            f"{CLASS_NAMES[class_id]}"
        )
        print("-" * 70)

        real_class_x = (
            target_x[
                target_y == class_id
            ]
        )

        if len(real_class_x) < 2:
            raise RuntimeError(
                f"Too few target trials for "
                f"class {class_id}: "
                f"{len(real_class_x)}"
            )

        (
            generator,
            d_phi,
            d_psi,
            history,
        ) = train_one_class_fbgan(
            real_class_x,
            sparse_fbcsp,
            cfg,
        )

        fake_class = (
            generate_fake_samples(
                generator,
                per_class,
                cfg,
            )
        )

        fake_parts.append(
            fake_class
        )

        fake_labels.append(
            np.full(
                per_class,
                class_id,
                dtype=np.int64,
            )
        )

        history = history.copy()
        history["class_id"] = class_id
        history["class_name"] = (
            CLASS_NAMES[class_id]
        )
        histories.append(
            history
        )

        del (
            generator,
            d_phi,
            d_psi,
            real_class_x,
            fake_class,
        )

        gc.collect()

        if DEVICE.type == "mps":
            torch.mps.empty_cache()

    fake_x = np.concatenate(
        fake_parts,
        axis=0,
    )

    fake_y = np.concatenate(
        fake_labels,
        axis=0,
    )

    history_df = pd.concat(
        histories,
        ignore_index=True,
    )

    assert fake_x.shape == (
        total,
        cfg.n_channels,
        cfg.n_times,
    )

    print(
        "\nGenerated maximum fake pool:",
        fake_x.shape,
    )

    print(
        "Fake class distribution:",
        np.bincount(
            fake_y,
            minlength=cfg.n_classes,
        ),
    )

    return (
        fake_x,
        fake_y,
        history_df,
    )


# ============================================================
# FBGAN SHAPE TESTS
# ============================================================

print("=" * 70)
print("V4 FBGAN SHAPE TEST")
print("=" * 70)

_shape_batch = 2

_shape_z = torch.randn(
    _shape_batch,
    CFG.latent_dim,
    device=DEVICE,
)

_shape_g = (
    FBGANGenerator(
        CFG.latent_dim
    )
    .to(DEVICE)
)

with torch.no_grad():
    _shape_fake = _shape_g(
        _shape_z,
        debug=True,
    )

assert _shape_fake.shape == (
    _shape_batch,
    1,
    64,
    640,
)

_shape_dphi = (
    RawEEGDiscriminator(
        CFG.n_times
    )
    .to(DEVICE)
)

with torch.no_grad():
    _shape_phi = _shape_dphi(
        _shape_fake,
        debug=True,
    )

assert _shape_phi.shape == (
    _shape_batch,
    1,
)

_shape_sparse_channels = 16

_shape_dpsi = (
    SparseFilterBankDiscriminator(
        _shape_sparse_channels,
        CFG.n_times,
    )
    .to(DEVICE)
)

_shape_sparse = torch.randn(
    _shape_batch,
    1,
    _shape_sparse_channels,
    640,
    device=DEVICE,
)

with torch.no_grad():
    _shape_psi = _shape_dpsi(
        _shape_sparse,
        debug=True,
    )

assert _shape_psi.shape == (
    _shape_batch,
    1,
)

print(
    "\nFBGAN V4 shape tests PASSED."
)

del (
    _shape_z,
    _shape_g,
    _shape_fake,
    _shape_dphi,
    _shape_phi,
    _shape_dpsi,
    _shape_sparse,
    _shape_psi,
)

gc.collect()

if DEVICE.type == "mps":
    torch.mps.empty_cache()


V4 FBGAN SHAPE TEST
G FC reshape: (2, 128, 25, 80)
G ConvTrans1: (2, 128, 27, 252)
G ConvTrans2: (2, 128, 29, 768)
G ConvTrans3: (2, 64, 31, 1539)
G ConvTrans4: (2, 32, 64, 1543)
G ConvTrans5: (2, 1, 64, 1544)
G PhysioNet output: (2, 1, 64, 640)
D_phi Conv1: (2, 10, 64, 618)
D_phi Conv2: (2, 30, 1, 618)
D_phi Conv3: (2, 30, 1, 602)
D_phi Pool1: (2, 30, 1, 100)
D_phi Conv4: (2, 30, 1, 14)
D_phi Pool2: (2, 30, 1, 2)
D_phi Flatten: (2, 60)
D_psi Conv1: (2, 10, 16, 618)
D_psi Conv2: (2, 30, 4, 618)
D_psi Conv3: (2, 30, 1, 618)
D_psi Conv4: (2, 30, 1, 602)
D_psi Pool1: (2, 30, 1, 100)
D_psi Conv5: (2, 30, 1, 94)
D_psi Pool2: (2, 30, 1, 15)
D_psi Flatten: (2, 450)

FBGAN V4 shape tests PASSED.


In [7]:

# ============================================================
# CELL 6 - V4 CRNN-DF CLASSIFIER
# ============================================================


class CRNNDF(nn.Module):
    """
    CRNN-DF adapted to PhysioNet.

    Published classifier design:
        spatial-temporal Conv kernel C x 45
        maxpool kernel 1 x 75, stride 10
        2-layer LSTM, hidden 64
        dropout 0.5

    PhysioNet:
        C = 64
        T = 640

    No padding is used in the first convolution, preserving the
    paper's temporal-window logic:
        640 - 45 + 1 = 596
        pool 75, stride 10 -> 53 time steps
    """

    def __init__(
        self,
        n_classes=4,
        hidden_size=64,
        dropout=0.5,
    ):
        super().__init__()

        self.conv = nn.Conv2d(
            1,
            40,
            kernel_size=(64, 45),
            stride=(1, 1),
            padding=(0, 0),
            bias=False,
        )

        self.batch_norm = nn.BatchNorm2d(
            40
        )

        self.activation = nn.ELU(
            inplace=True
        )

        self.pool = nn.MaxPool2d(
            kernel_size=(1, 75),
            stride=(1, 10),
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.lstm = nn.LSTM(
            input_size=40,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
        )

        self.classifier = nn.Linear(
            hidden_size,
            n_classes,
        )

    def forward(
        self,
        x,
        debug=False,
    ):
        if x.ndim == 3:
            x = x.unsqueeze(
                1
            )

        if debug:
            print(
                "CRNN input:",
                tuple(x.shape),
            )

        x = self.conv(
            x
        )

        x = self.batch_norm(
            x
        )

        x = self.activation(
            x
        )

        if debug:
            print(
                "CRNN spatial conv:",
                tuple(x.shape),
            )

        x = self.pool(
            x
        )

        if debug:
            print(
                "CRNN maxpool:",
                tuple(x.shape),
            )

        x = self.dropout(
            x
        )

        x = x.squeeze(
            2
        )

        x = x.transpose(
            1,
            2,
        )

        if debug:
            print(
                "CRNN LSTM sequence:",
                tuple(x.shape),
            )

        sequence, _ = self.lstm(
            x
        )

        features = sequence[
            :,
            -1,
            :,
        ]

        features = self.dropout(
            features
        )

        logits = self.classifier(
            features
        )

        if debug:
            print(
                "CRNN features:",
                tuple(features.shape),
            )
            print(
                "CRNN logits:",
                tuple(logits.shape),
            )

        return (
            logits,
            features,
        )


print(
    "=" * 70
)
print(
    "CRNN-DF SHAPE TEST"
)
print(
    "=" * 70
)

_test_crnn = (
    CRNNDF(
        CFG.n_classes
    )
    .to(DEVICE)
)

_test_input = torch.randn(
    2,
    1,
    CFG.n_channels,
    CFG.n_times,
    device=DEVICE,
)

with torch.no_grad():
    _test_logits, _test_features = (
        _test_crnn(
            _test_input,
            debug=True,
        )
    )

assert _test_logits.shape == (
    2,
    4,
)

assert _test_features.shape == (
    2,
    64,
)

print(
    "CRNN-DF V4 shape test PASSED."
)

del (
    _test_crnn,
    _test_input,
    _test_logits,
    _test_features,
)

gc.collect()

if DEVICE.type == "mps":
    torch.mps.empty_cache()


CRNN-DF SHAPE TEST
CRNN input: (2, 1, 64, 640)
CRNN spatial conv: (2, 40, 1, 596)
CRNN maxpool: (2, 40, 1, 53)
CRNN LSTM sequence: (2, 53, 40)
CRNN features: (2, 64)
CRNN logits: (2, 4)
CRNN-DF V4 shape test PASSED.


In [8]:
# CELL 7 - Discriminative Feature Loss + Center Initialization
# ============================================================


class CenterDistanceLoss(nn.Module):
    """
    Central-distance loss from the paper.

    L_cen = mean ||v_i - center_yi||_2

    The center vectors are initialized from the training features and
    updated every 15 epochs with alpha=0.02.
    """

    def __init__(
        self,
        n_classes=4,
        feature_dim=64,
    ):
        super().__init__()

        self.register_buffer(
            "centers",
            torch.zeros(
                n_classes,
                feature_dim,
            ),
        )

    @torch.no_grad()
    def initialize(
        self,
        features,
        labels,
    ):
        for class_id in range(
            self.centers.shape[0]
        ):
            mask = labels == class_id

            if mask.any():
                self.centers[
                    class_id
                ] = features[
                    mask
                ].mean(dim=0)

    def forward(
        self,
        features,
        labels,
    ):
        centers = self.centers[
            labels
        ]

        distances = torch.norm(
            features - centers,
            p=2,
            dim=1,
        )

        return distances.mean()

    @torch.no_grad()
    def shift_centers(
        self,
        features,
        labels,
        alpha=0.02,
    ):
        # First update centers toward their current batch means.
        for class_id in range(
            self.centers.shape[0]
        ):
            mask = labels == class_id

            if mask.any():
                batch_center = (
                    features[mask]
                    .mean(dim=0)
                )

                self.centers[
                    class_id
                ] = (
                    (1.0 - alpha)
                    * self.centers[
                        class_id
                    ]
                    + alpha
                    * batch_center
                )

        # Then expand inter-class center separation.
        global_center = (
            self.centers.mean(dim=0)
        )

        directions = (
            self.centers
            - global_center
        )

        norms = torch.norm(
            directions,
            dim=1,
            keepdim=True,
        ).clamp_min(1e-8)

        self.centers += (
            alpha
            * directions
            / norms
        )


def joint_loss(
    logits,
    features,
    labels,
    center_loss,
    lambda_center=0.1,
):
    classification_loss = (
        F.cross_entropy(
            logits,
            labels,
        )
    )

    center_distance = center_loss(
        features,
        labels,
    )

    total = (
        classification_loss
        + lambda_center
        * center_distance
    )

    return (
        total,
        classification_loss.detach(),
        center_distance.detach(),
    )


class EEGDataset(Dataset):
    def __init__(
        self,
        x,
        y,
    ):
        self.x = torch.tensor(
            x,
            dtype=torch.float32,
        )
        self.y = torch.tensor(
            y,
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(
        self,
        index,
    ):
        return (
            self.x[index],
            self.y[index],
        )


def make_source_train_val_split(
    x,
    y,
    cfg=CFG,
):
    indices = np.arange(
        len(y)
    )

    train_idx, val_idx = (
        train_test_split(
            indices,
            test_size=cfg.source_validation_fraction,
            random_state=SEED,
            stratify=y,
        )
    )

    return (
        train_idx,
        val_idx,
    )


In [ ]:

# ============================================================
# CELL 8 - V4 FULL-SOURCE LOSO + AUGMENTATION ABLATION
# ============================================================
#
# This is the critical V4 correction.
#
# For each target subject:
#
#     target = 1 subject
#     source = ALL other available subjects
#
# Therefore, with 109 subjects:
#
#     target = 1
#     source = 108
#
# Five targets means five folds, each with ~108 source subjects.
#
# FBGAN is trained ONCE per target at the maximum 3000 samples.
# Its balanced fake pool is then reused for:
#
#     0
#     500
#     1000
#     2000
#     3000
#
# so the augmentation ablation does not require retraining the GAN
# five times per target.
# ============================================================


def load_subject_collection(
    subject_ids,
):
    """
    Load a list of subjects into a single source array.

    For one V4 fold this is the full source population excluding
    the held-out target.
    """

    x_parts = []
    y_parts = []
    subject_parts = []

    for subject_id in tqdm(
        subject_ids,
        desc="Loading source subjects",
    ):
        x_subject, y_subject = (
            load_subject_data(
                subject_id
            )
        )

        if x_subject is None:
            continue

        x_parts.append(
            x_subject
        )

        y_parts.append(
            y_subject
        )

        subject_parts.append(
            np.full(
                len(y_subject),
                subject_id,
                dtype=np.int64,
            )
        )

        del (
            x_subject,
            y_subject,
        )

        gc.collect()

    if not x_parts:
        raise RuntimeError(
            "No source subjects could be loaded."
        )

    return (
        np.concatenate(
            x_parts,
            axis=0,
        ),
        np.concatenate(
            y_parts,
            axis=0,
        ),
        np.concatenate(
            subject_parts,
            axis=0,
        ),
    )


class TrainZScore:
    """
    Source-only standardization.

    Statistics are learned ONLY from source subjects.
    """

    def fit(
        self,
        x,
    ):
        self.mean_ = x.mean(
            axis=(0, 2),
            keepdims=True,
        )

        self.std_ = np.sqrt(
            x.var(
                axis=(0, 2),
                keepdims=True,
            )
            + 1e-8
        )

        return self

    def transform(
        self,
        x,
    ):
        return (
            (
                x
                - self.mean_
            )
            / self.std_
        ).astype(
            np.float32
        )

    def fit_transform(
        self,
        x,
    ):
        return self.fit(
            x
        ).transform(
            x
        )


class EEGDataset(Dataset):
    def __init__(
        self,
        x,
        y,
    ):
        self.x = torch.tensor(
            x,
            dtype=torch.float32,
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long,
        )

    def __len__(
        self,
    ):
        return len(
            self.y
        )

    def __getitem__(
        self,
        index,
    ):
        return (
            self.x[index],
            self.y[index],
        )


def source_train_val_split(
    x,
    y,
    cfg,
):
    indices = np.arange(
        len(y)
    )

    train_idx, val_idx = (
        train_test_split(
            indices,
            test_size=cfg.source_validation_fraction,
            random_state=SEED,
            stratify=y,
        )
    )

    return (
        train_idx,
        val_idx,
    )


def train_classifier(
    x_train,
    y_train,
    x_val,
    y_val,
    cfg,
):
    train_loader = DataLoader(
        EEGDataset(
            x_train,
            y_train,
        ),
        batch_size=cfg.classifier_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
    )

    val_loader = DataLoader(
        EEGDataset(
            x_val,
            y_val,
        ),
        batch_size=cfg.classifier_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
    )

    model = (
        CRNNDF(
            cfg.n_classes
        )
        .to(DEVICE)
    )

    center_loss = CenterDistanceLoss(
        n_classes=cfg.n_classes,
        feature_dim=64,
    ).to(DEVICE)

    # --------------------------------------------------------
    # Initialize centers from source-training features.
    # --------------------------------------------------------

    model.eval()

    init_loader = DataLoader(
        EEGDataset(
            x_train,
            y_train,
        ),
        batch_size=cfg.classifier_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
    )

    init_features = []
    init_labels = []

    with torch.no_grad():
        for x_batch, y_batch in init_loader:
            x_batch = x_batch.to(
                DEVICE
            )

            _, features = model(
                x_batch
            )

            init_features.append(
                features
            )

            init_labels.append(
                y_batch.to(DEVICE)
            )

    center_loss.initialize(
        torch.cat(
            init_features
        ),
        torch.cat(
            init_labels
        ),
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.classifier_lr,
        weight_decay=cfg.classifier_weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,
        )
    )

    best_val = -np.inf
    best_state = None
    bad_epochs = 0
    history = []

    for epoch in range(
        1,
        cfg.classifier_epochs + 1,
    ):
        model.train()

        running_loss = 0.0

        epoch_features = []
        epoch_labels = []

        for x_batch, y_batch in (
            train_loader
        ):
            x_batch = x_batch.to(
                DEVICE
            )

            y_batch = y_batch.to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits, features = model(
                x_batch
            )

            classification_loss = (
                F.cross_entropy(
                    logits,
                    y_batch,
                )
            )

            center_distance = (
                center_loss(
                    features,
                    y_batch,
                )
            )

            total_loss = (
                classification_loss
                + cfg.lambda_center
                * center_distance
            )

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.classifier_gradient_clip,
            )

            optimizer.step()

            running_loss += (
                total_loss.item()
                * len(y_batch)
            )

            epoch_features.append(
                features.detach()
            )

            epoch_labels.append(
                y_batch.detach()
            )

        # Paper: update center vectors every 15 epochs.
        if (
            epoch
            % cfg.center_update_every
            == 0
        ):
            center_loss.shift_centers(
                torch.cat(
                    epoch_features
                ),
                torch.cat(
                    epoch_labels
                ),
                alpha=cfg.center_alpha,
            )

        # ----------------------------------------------------
        # Source-only validation
        # ----------------------------------------------------

        model.eval()

        val_predictions = []
        val_targets = []

        with torch.no_grad():
            for x_batch, y_batch in (
                val_loader
            ):
                x_batch = x_batch.to(
                    DEVICE
                )

                logits, _ = model(
                    x_batch
                )

                val_predictions.extend(
                    logits.argmax(
                        dim=1
                    )
                    .cpu()
                    .numpy()
                )

                val_targets.extend(
                    y_batch.numpy()
                )

        val_accuracy = accuracy_score(
            val_targets,
            val_predictions,
        )

        scheduler.step(
            val_accuracy
        )

        mean_train_loss = (
            running_loss
            / len(
                train_loader.dataset
            )
        )

        history.append(
            {
                "epoch": epoch,
                "train_loss": mean_train_loss,
                "source_val_accuracy": (
                    val_accuracy
                ),
                "lr": optimizer.param_groups[0][
                    "lr"
                ],
            }
        )

        if (
            epoch == 1
            or epoch % 5 == 0
            or val_accuracy > best_val
        ):
            print(
                f"Epoch {epoch:03d}/"
                f"{cfg.classifier_epochs} | "
                f"loss={mean_train_loss:.4f} | "
                f"source_val="
                f"{val_accuracy:.4f} | "
                f"lr="
                f"{optimizer.param_groups[0]['lr']:.2e}"
            )

        if val_accuracy > best_val:
            best_val = val_accuracy
            bad_epochs = 0

            best_state = {
                key: value.detach()
                .cpu()
                .clone()
                for key, value
                in model.state_dict().items()
            }

        else:
            bad_epochs += 1

            if (
                bad_epochs
                >= cfg.classifier_patience
            ):
                print(
                    "Early stopping at epoch",
                    epoch,
                )
                break

    if best_state is None:
        raise RuntimeError(
            "No classifier checkpoint was produced."
        )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(
            history
        ),
    )


@torch.no_grad()
def predict_model(
    model,
    x,
    cfg,
):
    loader = DataLoader(
        EEGDataset(
            x,
            np.zeros(
                len(x),
                dtype=np.int64,
            ),
        ),
        batch_size=cfg.classifier_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
    )

    model.eval()

    predictions = []
    features = []

    for x_batch, _ in loader:
        x_batch = x_batch.to(
            DEVICE
        )

        logits, feature = model(
            x_batch
        )

        predictions.append(
            logits.argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )

        features.append(
            feature.cpu().numpy()
        )

    return (
        np.concatenate(
            predictions
        ),
        np.concatenate(
            features
        ),
    )


def evaluate_gan_quality(
    real_x,
    fake_x,
    sfreq=160,
):
    """
    Lightweight quality diagnostics:
        amplitude scale
        PSD correlation
        covariance RMSE
    """

    real_std = real_x.std(
        axis=(0, 2)
    )

    fake_std = fake_x.std(
        axis=(0, 2)
    )

    mean_abs_scale_diff = float(
        np.mean(
            np.abs(
                real_std
                - fake_std
            )
        )
    )

    real_avg = (
        real_x.mean(
            axis=1
        )
        .reshape(-1)
    )

    fake_avg = (
        fake_x.mean(
            axis=1
        )
        .reshape(-1)
    )

    fr, pr = welch(
        real_avg,
        fs=sfreq,
        nperseg=256,
    )

    ff, pf = welch(
        fake_avg,
        fs=sfreq,
        nperseg=256,
    )

    mask_r = (
        (fr >= 1)
        & (fr <= 38)
    )

    mask_f = (
        (ff >= 1)
        & (ff <= 38)
    )

    pr = np.log10(
        pr[mask_r]
        + 1e-12
    )

    pf = np.log10(
        pf[mask_f]
        + 1e-12
    )

    n = min(
        len(pr),
        len(pf),
    )

    psd_corr = float(
        np.corrcoef(
            pr[:n],
            pf[:n],
        )[0, 1]
    )

    def mean_covariance(
        x,
    ):
        values = []

        for trial in x[
            : min(
                64,
                len(x),
            )
        ]:
            cov = (
                trial @ trial.T
            )

            cov = (
                cov
                / (
                    np.trace(cov)
                    + 1e-8
                )
            )

            values.append(
                cov
            )

        return np.mean(
            values,
            axis=0,
        )

    real_cov = mean_covariance(
        real_x
    )

    fake_cov = mean_covariance(
        fake_x
    )

    covariance_rmse = float(
        np.sqrt(
            np.mean(
                (
                    real_cov
                    - fake_cov
                )
                ** 2
            )
        )
    )

    return {
        "mean_abs_scale_diff": (
            mean_abs_scale_diff
        ),
        "psd_correlation": psd_corr,
        "covariance_rmse": (
            covariance_rmse
        ),
    }


def balanced_fake_subset(
    fake_x,
    fake_y,
    total_fake,
    n_classes=4,
):
    """
    Return exactly total_fake samples, balanced across classes.

    The maximum generated pool is 3000.
    """

    if total_fake == 0:
        return (
            np.empty(
                (
                    0,
                    fake_x.shape[1],
                    fake_x.shape[2],
                ),
                dtype=np.float32,
            ),
            np.empty(
                (
                    0,
                ),
                dtype=np.int64,
            ),
        )

    if total_fake % n_classes != 0:
        raise ValueError(
            "Augmentation total must be divisible by 4."
        )

    per_class = (
        total_fake
        // n_classes
    )

    parts_x = []
    parts_y = []

    for class_id in range(
        n_classes
    ):
        idx = np.flatnonzero(
            fake_y == class_id
        )

        if len(idx) < per_class:
            raise RuntimeError(
                "Not enough fake samples for "
                f"class {class_id}."
            )

        # Because the maximum pool is generated in class order,
        # deterministic first-per-class selection gives a stable
        # ablation.
        selected = idx[
            :per_class
        ]

        parts_x.append(
            fake_x[selected]
        )

        parts_y.append(
            fake_y[selected]
        )

    return (
        np.concatenate(
            parts_x,
            axis=0,
        ),
        np.concatenate(
            parts_y,
            axis=0,
        ),
    )


def run_v4_loso():
    fold_rows = []
    ablation_rows = []
    prediction_rows = []
    history_rows = []
    gan_quality_rows = []

    for target_subject in TARGET_SUBJECTS:

        print("\n" + "=" * 78)
        print(
            f"V4 LOSO TARGET: "
            f"S{target_subject:03d}"
        )
        print("=" * 78)

        source_subjects = [
            sid
            for sid in ALL_SUBJECTS
            if sid != target_subject
        ]

        if len(source_subjects) < 2:
            raise RuntimeError(
                "Insufficient source subjects."
            )

        # ----------------------------------------------------
        # Load source population and target subject.
        # ----------------------------------------------------

        source_x_raw, source_y, source_subject_ids = (
            load_subject_collection(
                source_subjects
            )
        )

        target_x_raw, target_y = (
            load_subject_data(
                target_subject
            )
        )

        if target_x_raw is None:
            raise RuntimeError(
                f"Target S{target_subject:03d} unavailable."
            )

        print(
            "\nSOURCE DATA:",
            source_x_raw.shape,
        )

        print(
            "TARGET DATA:",
            target_x_raw.shape,
        )

        # ----------------------------------------------------
        # Source-only normalization.
        # ----------------------------------------------------

        normalizer = TrainZScore()

        source_x = (
            normalizer.fit_transform(
                source_x_raw
            )
        )

        target_x = (
            normalizer.transform(
                target_x_raw
            )
        )

        # ----------------------------------------------------
        # Target adaptation data.
        # ----------------------------------------------------

        if (
            CFG.paper_faithful_target_adaptation
        ):
            target_adapt_x = target_x
            target_adapt_y = target_y

            x_test = target_x
            y_test = target_y

            print(
                "\nTARGET ADAPTATION: "
                "paper-faithful"
            )

            print(
                "Target EEG is used for FBGAN adaptation."
            )

        else:
            adapt_idx, test_idx = (
                train_test_split(
                    np.arange(
                        len(target_y)
                    ),
                    test_size=(
                        1.0
                        - CFG.target_calibration_fraction
                    ),
                    random_state=SEED,
                    stratify=target_y,
                )
            )

            target_adapt_x = target_x[
                adapt_idx
            ]

            target_adapt_y = target_y[
                adapt_idx
            ]

            x_test = target_x[
                test_idx
            ]

            y_test = target_y[
                test_idx
            ]

            print(
                "\nTARGET ADAPTATION: "
                "strict calibration/test"
            )

        # ----------------------------------------------------
        # Target FBCSP + LASSO
        # ----------------------------------------------------

        print(
            "\nFitting target FBCSP + LASSO..."
        )

        sparse_fbcsp = fit_sparse_fbcsp(
            target_adapt_x,
            target_adapt_y,
            CFG,
        )

        target_sparse = (
            sparse_fbcsp.transform(
                target_adapt_x,
                CFG.sfreq,
            )
        )

        n_sparse = len(
            sparse_fbcsp.selected_indices_
        )

        print(
            "Target sparse shape:",
            target_sparse.shape,
        )

        print(
            "Selected sparse components:",
            n_sparse,
            "/ 160",
        )

        # ----------------------------------------------------
        # FBGAN maximum pool
        # ----------------------------------------------------

        print(
            "\nTraining one FBGAN per target class..."
        )

        (
            fake_x,
            fake_y,
            gan_history,
        ) = generate_balanced_fake_pool(
            target_adapt_x,
            target_adapt_y,
            sparse_fbcsp,
            CFG,
        )

        quality = evaluate_gan_quality(
            target_adapt_x,
            fake_x,
            CFG.sfreq,
        )

        quality[
            "test_subject"
        ] = int(
            target_subject
        )

        quality[
            "n_sparse_features"
        ] = int(
            n_sparse
        )

        gan_quality_rows.append(
            quality
        )

        print(
            "GAN quality:",
            quality,
        )

        # ----------------------------------------------------
        # Source train/validation split.
        # ----------------------------------------------------

        (
            source_train_idx,
            source_val_idx,
        ) = source_train_val_split(
            source_x,
            source_y,
            CFG,
        )

        source_train_x = (
            source_x[
                source_train_idx
            ]
        )

        source_train_y = (
            source_y[
                source_train_idx
            ]
        )

        source_val_x = (
            source_x[
                source_val_idx
            ]
        )

        source_val_y = (
            source_y[
                source_val_idx
            ]
        )

        print(
            "\nSource train trials:",
            len(source_train_y),
        )

        print(
            "Source validation trials:",
            len(source_val_y),
        )

        # ----------------------------------------------------
        # Augmentation ablation
        # ----------------------------------------------------

        for total_fake in (
            CFG.augmentation_levels
        ):

            print("\n" + "-" * 78)
            print(
                f"TARGET S{target_subject:03d} | "
                f"FAKE SAMPLES = {total_fake}"
            )
            print("-" * 78)

            (
                selected_fake_x,
                selected_fake_y,
            ) = balanced_fake_subset(
                fake_x,
                fake_y,
                total_fake,
                CFG.n_classes,
            )

            if total_fake == 0:
                train_x = (
                    source_train_x
                )

                train_y = (
                    source_train_y
                )

            else:
                train_x = (
                    np.concatenate(
                        [
                            source_train_x,
                            selected_fake_x,
                        ],
                        axis=0,
                    )
                )

                train_y = (
                    np.concatenate(
                        [
                            source_train_y,
                            selected_fake_y,
                        ],
                        axis=0,
                    )
                )

            print(
                "Classifier training shape:",
                train_x.shape,
            )

            (
                model,
                history,
            ) = train_classifier(
                train_x,
                train_y,
                source_val_x,
                source_val_y,
                CFG,
            )

            predictions, features = (
                predict_model(
                    model,
                    x_test,
                    CFG,
                )
            )

            accuracy = accuracy_score(
                y_test,
                predictions,
            )

            balanced = (
                balanced_accuracy_score(
                    y_test,
                    predictions,
                )
            )

            print(
                f"S{target_subject:03d} | "
                f"fake={total_fake:4d} | "
                f"accuracy={accuracy:.4f} | "
                f"balanced={balanced:.4f}"
            )

            ablation_rows.append(
                {
                    "test_subject": int(
                        target_subject
                    ),
                    "fake_samples": int(
                        total_fake
                    ),
                    "accuracy": float(
                        accuracy
                    ),
                    "balanced_accuracy": float(
                        balanced
                    ),
                    "n_test": int(
                        len(y_test)
                    ),
                    "n_source_train": int(
                        len(source_train_y)
                    ),
                    "n_train_total": int(
                        len(train_y)
                    ),
                    "n_sparse_features": int(
                        n_sparse
                    ),
                }
            )

            for (
                true_label,
                predicted_label,
                feature_vector,
            ) in zip(
                y_test,
                predictions,
                features,
            ):
                prediction_rows.append(
                    {
                        "test_subject": int(
                            target_subject
                        ),
                        "fake_samples": int(
                            total_fake
                        ),
                        "true": int(
                            true_label
                        ),
                        "pred": int(
                            predicted_label
                        ),
                        "feature": feature_vector,
                    }
                )

            history_rows.append(
                history.assign(
                    test_subject=int(
                        target_subject
                    ),
                    fake_samples=int(
                        total_fake
                    ),
                )
            )

            del (
                model,
                train_x,
                train_y,
                selected_fake_x,
                selected_fake_y,
                predictions,
                features,
            )

            gc.collect()

            if DEVICE.type == "mps":
                torch.mps.empty_cache()

        fold_rows.append(
            {
                "test_subject": int(
                    target_subject
                ),
                "n_source_subjects": int(
                    len(source_subjects)
                ),
                "n_source_trials": int(
                    len(source_y)
                ),
                "n_target_trials": int(
                    len(y_test)
                ),
                "n_sparse_features": int(
                    n_sparse
                ),
            }
        )

        # ----------------------------------------------------
        # Fold cleanup.
        # ----------------------------------------------------

        del (
            source_x_raw,
            source_y,
            source_subject_ids,
            target_x_raw,
            target_y,
            source_x,
            target_x,
            target_adapt_x,
            target_adapt_y,
            fake_x,
            fake_y,
            gan_history,
            sparse_fbcsp,
            source_train_x,
            source_train_y,
            source_val_x,
            source_val_y,
        )

        gc.collect()

        if DEVICE.type == "mps":
            torch.mps.empty_cache()

    fold_df = pd.DataFrame(
        fold_rows
    )

    ablation_df = pd.DataFrame(
        ablation_rows
    )

    predictions_df = pd.DataFrame(
        prediction_rows
    )

    history_df = pd.concat(
        history_rows,
        ignore_index=True,
    )

    gan_quality_df = pd.DataFrame(
        gan_quality_rows
    )

    summary_by_augmentation = (
        ablation_df.groupby(
            "fake_samples"
        )[
            [
                "accuracy",
                "balanced_accuracy",
            ]
        ]
        .agg(
            [
                "mean",
                "std",
            ]
        )
        .reset_index()
    )

    return (
        fold_df,
        ablation_df,
        predictions_df,
        history_df,
        gan_quality_df,
        summary_by_augmentation,
    )


# ============================================================
# RUN V4
# ============================================================

print(
    "Starting V4 full-source LOSO + augmentation ablation."
)

(
    FOLD_DF,
    ABLATION_DF,
    PREDICTIONS_DF,
    HISTORY_DF,
    GAN_QUALITY_DF,
    ABLATION_SUMMARY,
) = run_v4_loso()

print(
    "\n" + "=" * 78
)
print(
    "V4 AUGMENTATION SUMMARY"
)
print(
    "=" * 78
)

print(
    ABLATION_SUMMARY.to_string(
        index=False
    )
)


In [ ]:

# ============================================================
# CELL 9 - V4 EVALUATION AND ABLATION ANALYSIS
# ============================================================


def print_subject_ablation_table(
    ablation_df,
):
    pivot = (
        ablation_df
        .pivot(
            index="test_subject",
            columns="fake_samples",
            values="accuracy",
        )
        .sort_index()
    )

    print(
        pivot.to_string(
            float_format=lambda value:
            f"{value:.4f}"
        )
    )


def plot_augmentation_summary(
    ablation_df,
):
    summary = (
        ablation_df
        .groupby(
            "fake_samples"
        )
        .agg(
            mean_accuracy=(
                "accuracy",
                "mean",
            ),
            std_accuracy=(
                "accuracy",
                "std",
            ),
            mean_balanced_accuracy=(
                "balanced_accuracy",
                "mean",
            ),
        )
        .reset_index()
    )

    plt.figure(
        figsize=(9, 5)
    )

    plt.errorbar(
        summary[
            "fake_samples"
        ],
        summary[
            "mean_accuracy"
        ],
        yerr=summary[
            "std_accuracy"
        ],
        marker="o",
        capsize=4,
        label="Accuracy",
    )

    plt.axhline(
        0.25,
        linestyle="--",
        linewidth=1.2,
        label="4-class chance",
    )

    plt.xlabel(
        "Number of generated samples"
    )

    plt.ylabel(
        "Mean LOSO accuracy"
    )

    plt.title(
        "V4 FBGAN augmentation ablation"
    )

    plt.legend()
    plt.tight_layout()
    plt.show()

    return summary


def plot_subject_accuracy(
    ablation_df,
):
    subjects = sorted(
        ablation_df[
            "test_subject"
        ].unique()
    )

    levels = sorted(
        ablation_df[
            "fake_samples"
        ].unique()
    )

    x = np.arange(
        len(subjects)
    )

    width = (
        0.8
        / max(
            len(levels),
            1,
        )
    )

    plt.figure(
        figsize=(12, 5)
    )

    for i, level in enumerate(
        levels
    ):
        subset = (
            ablation_df[
                ablation_df[
                    "fake_samples"
                ]
                == level
            ]
            .set_index(
                "test_subject"
            )
            .reindex(
                subjects
            )
        )

        plt.bar(
            x + i * width,
            subset[
                "accuracy"
            ],
            width=width,
            label=str(level),
        )

    plt.axhline(
        0.25,
        linestyle="--",
        linewidth=1.0,
    )

    plt.xticks(
        x
        + width
        * (len(levels) - 1)
        / 2,
        [
            str(subject)
            for subject in subjects
        ],
    )

    plt.xlabel(
        "Held-out target subject"
    )

    plt.ylabel(
        "Accuracy"
    )

    plt.title(
        "V4 per-subject augmentation results"
    )

    plt.legend(
        title="Fake samples"
    )

    plt.tight_layout()
    plt.show()


def print_best_development_setting(
    ablation_df,
):
    summary = (
        ablation_df
        .groupby(
            "fake_samples"
        )[
            "accuracy"
        ]
        .agg(
            [
                "mean",
                "std",
            ]
        )
        .reset_index()
    )

    best = summary.iloc[
        summary[
            "mean"
        ].argmax()
    ]

    print(
        "\nBest DEVELOPMENT augmentation:"
    )

    print(
        f"{int(best['fake_samples'])} fake samples"
    )

    print(
        f"Mean accuracy = "
        f"{best['mean']:.4f}"
    )

    print(
        f"Std accuracy = "
        f"{best['std']:.4f}"
    )

    return int(
        best["fake_samples"]
    )


def evaluate_selected_level(
    predictions_df,
    selected_fake_samples,
):
    subset = predictions_df[
        predictions_df[
            "fake_samples"
        ]
        == selected_fake_samples
    ]

    y_true = subset[
        "true"
    ].to_numpy()

    y_pred = subset[
        "pred"
    ].to_numpy()

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                CLASS_NAMES[i]
                for i in range(
                    CFG.n_classes
                )
            ],
            zero_division=0,
        )
    )

    matrix = confusion_matrix(
        y_true,
        y_pred,
    )

    plt.figure(
        figsize=(7, 6)
    )

    plt.imshow(
        matrix
    )

    plt.colorbar()

    plt.xticks(
        range(
            CFG.n_classes
        ),
        [
            CLASS_NAMES[i]
            for i in range(
                CFG.n_classes
            )
        ],
        rotation=30,
        ha="right",
    )

    plt.yticks(
        range(
            CFG.n_classes
        ),
        [
            CLASS_NAMES[i]
            for i in range(
                CFG.n_classes
            )
        ],
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.title(
        f"Confusion matrix — "
        f"{selected_fake_samples} fake samples"
    )

    for row in range(
        matrix.shape[0]
    ):
        for col in range(
            matrix.shape[1]
        ):
            plt.text(
                col,
                row,
                str(
                    matrix[
                        row,
                        col,
                    ]
                ),
                ha="center",
                va="center",
            )

    plt.tight_layout()
    plt.show()


def plot_tsne_for_level(
    predictions_df,
    fake_samples,
):
    subset = predictions_df[
        predictions_df[
            "fake_samples"
        ]
        == fake_samples
    ]

    if len(subset) < 10:
        print(
            "Too few samples for t-SNE."
        )
        return

    features = np.stack(
        subset[
            "feature"
        ].to_numpy()
    )

    labels = subset[
        "true"
    ].to_numpy()

    perplexity = min(
        30,
        max(
            5,
            len(features) // 10,
        ),
        len(features) - 1,
    )

    embedding = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=SEED,
    ).fit_transform(
        features
    )

    plt.figure(
        figsize=(9, 7)
    )

    for class_id in range(
        CFG.n_classes
    ):
        mask = (
            labels == class_id
        )

        plt.scatter(
            embedding[
                mask,
                0,
            ],
            embedding[
                mask,
                1,
            ],
            label=CLASS_NAMES[
                class_id
            ],
            alpha=0.70,
        )

    plt.title(
        f"CRNN-DF features — "
        f"{fake_samples} fake samples"
    )

    plt.xlabel(
        "t-SNE 1"
    )

    plt.ylabel(
        "t-SNE 2"
    )

    plt.legend()
    plt.tight_layout()
    plt.show()


print(
    "=" * 78
)
print(
    "V4 PER-SUBJECT ABLATION TABLE"
)
print(
    "=" * 78
)

print_subject_ablation_table(
    ABLATION_DF
)

summary = plot_augmentation_summary(
    ABLATION_DF
)

plot_subject_accuracy(
    ABLATION_DF
)

BEST_DEVELOPMENT_FAKE_COUNT = (
    print_best_development_setting(
        ABLATION_DF
    )
)

print(
    "\nSelected development level:",
    BEST_DEVELOPMENT_FAKE_COUNT,
)

evaluate_selected_level(
    PREDICTIONS_DF,
    BEST_DEVELOPMENT_FAKE_COUNT,
)

plot_tsne_for_level(
    PREDICTIONS_DF,
    BEST_DEVELOPMENT_FAKE_COUNT,
)

print(
    "\nGAN quality diagnostics:"
)

print(
    GAN_QUALITY_DF.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Save all V4 experimental outputs.
# ------------------------------------------------------------

RESULTS_DIR = Path(
    "./v4_results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ABLATION_DF.to_csv(
    RESULTS_DIR
    / "v4_ablation_results.csv",
    index=False,
)

GAN_QUALITY_DF.to_csv(
    RESULTS_DIR
    / "v4_gan_quality.csv",
    index=False,
)

FOLD_DF.to_csv(
    RESULTS_DIR
    / "v4_fold_summary.csv",
    index=False,
)

print(
    "\nResults saved to:",
    RESULTS_DIR.resolve(),
)



## Research references used by V4

Zhang et al. (2023), *Subject-independent EEG classification based
on a hybrid neural network*, Frontiers in Neuroscience.

The paper reports:
- CRNN-DF: 63.52 +/- 10.70% on BCI Competition IV-2a.
- 500 fake samples: 68.53 +/- 10.55%.
- 1,000 fake samples: 69.90 +/- 10.97%.
- 2,000 fake samples: 71.31 +/- 11.42%.
- 3,000 fake samples: 72.82 +/- 10.44%.

The published generator table specifies:
- FC: 1600 -> 256000.
- ConvTrans1: 128 -> 128, kernel (3,15), stride (1,3).
- ConvTrans2: 128 -> 128, kernel (3,15), stride (1,3).
- ConvTrans3: 128 -> 64, kernel (3,5), stride (1,2).
- ConvTrans4: 64 -> 32, kernel (4,5), stride (2,1).
- ConvTrans5: 32 -> 1, kernel (1,2), stride (1,1).

The published discriminator table specifies D_phi and D_psi as the
raw-EEG and sparse-filter-bank discriminators. V4 adapts the spatial
kernel and the temporal geometry to 64 channels and 640 samples.

The paper's CRNN-DF description specifies a spatial-temporal
convolution with kernel C x 45, max pooling of 1 x 75 with stride 10,
a two-layer LSTM with hidden state 64, and dropout 0.5.

Source:
https://www.frontiersin.org/journals/neuroscience/articles/10.3389/fnins.2023.1124089/full
